# PropRAG vs GraphRAG vs BaseRAG — 2WikiMultiHopQA on a free Colab GPU

Benchmarks three RAG systems that **share one corpus, one embedder and one chat model**,
so the only thing compared is retrieval strategy and its cost.

* **Chat model:** `gpt-oss-20b` (Q4_K_M GGUF) via `llama-cpp-python`
* **Embedder:** `nvidia/NV-Embed-v2` (7.85B), loaded in **8-bit** to fit a T4

### Why three phases?
A free **T4 has 15 GB** of VRAM. NV-Embed-v2 (fp16 ≈ 16 GB) and the GGUF chat model
(≈ 12 GB) **cannot both be resident**. So the run is split so that each model is
loaded at most once and the two are **never** in VRAM together:

| Phase | Model resident | Work |
|---|---|---|
| **A** | chat LLM | extraction only (PropRAG NER+propositions, GraphRAG entities/relations/reports) |
| **B** | embedder | build all vector stores + graphs, run **all retrieval** (no LLM), score Recall@k |
| **C** | chat LLM | answer each question from its stored context, score EM/F1, build report |

Everything is written to **Google Drive** and every phase is **resume-safe** — a
disconnect never loses completed work; just re-run the phase cell.

> **Runtime:** set **Runtime → Change runtime type → T4 GPU** before running.
> NV-Embed-v2 is a **gated** model — accept its license at
> https://huggingface.co/nvidia/NV-Embed-v2 with the same account as your token.

## 1 · Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU."
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB")

## 2 · Install dependencies

`transformers`/`sentence-transformers` are pinned to versions known to run
NV-Embed-v2's custom code. `llama-cpp-python` is installed with a CUDA wheel
(falls back to a source build with CUDA if the wheel is unavailable — that path
takes ~5–10 min).

In [ ]:
%%bash
set -e
pip -q install "transformers==4.42.4" "sentence-transformers==2.7.0" accelerate bitsandbytes einops
pip -q install python-igraph pyarrow "huggingface_hub>=0.23"
# llama-cpp-python with CUDA: try a prebuilt wheel, else build from source with CUDA.
pip -q install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
  || CMAKE_ARGS="-DGGML_CUDA=on" pip -q install --no-cache-dir --force-reinstall llama-cpp-python
echo "deps installed"


## 3 · Hugging Face login

Paste a token (https://huggingface.co/settings/tokens) that has **accepted the
NV-Embed-v2 license**. It is used to download the gated embedder and the GGUF model.

In [ ]:
import os
from getpass import getpass
from huggingface_hub import login
tok = getpass("HF token: ").strip()
login(token=tok)
os.environ["HF_TOKEN"] = tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = tok
print("logged in")

## 4 · Mount Google Drive (persistent storage)

Indexes, embedding/LLM caches and results live here, so a disconnect doesn't
lose work. The code itself is written fresh to `/content` from this notebook.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/proprag_benchmark"   # <- change if you like
os.makedirs(PROJECT_DIR, exist_ok=True)
os.environ["PROPRAG_PROJECT_DIR"] = PROJECT_DIR
print("data + results ->", os.path.join(PROJECT_DIR, "data"))

## 5 · Write the benchmark code

The next cell creates the package directories; the cells after it write every
module. This is what makes the notebook self-contained.

In [ ]:
import os, sys
for d in ["proprag_poc/core", "proprag_poc/embedding", "proprag_poc/llm",
          "benchmark/graphrag"]:
    os.makedirs(os.path.join("/content", d), exist_ok=True)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
print("package dirs ready")

In [ ]:
%%writefile /content/proprag_poc/__init__.py
"""PropRAG-PDF POC: proposition knowledge-graph RAG over arbitrary PDFs."""

__version__ = "0.1.0"


In [ ]:
%%writefile /content/proprag_poc/config.py
"""Single configuration object for the PropRAG-PDF POC.

All tunable parameters live here. Pass one ``POCConfig`` instance everywhere;
never hardcode parameters in the pipeline modules.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Literal, Optional


# Backend presets: name -> (base_url, default model, needs_real_key).
# "google" uses the native google-genai SDK (Gemini); all others are OpenAI-compatible.
# "nvidia" is the free NVIDIA-hosted API (integrate.api.nvidia.com), OpenAI-compatible.
_LLM_PRESETS = {
    "nvidia": ("https://integrate.api.nvidia.com/v1", "meta/llama-3.3-70b-instruct", True),
    "google": (None, "gemini-2.5-flash", True),
    "koboldcpp": ("http://localhost:5001/v1", "gpt-oss-20b-Q4_K_M.gguf", False),
    "ollama": ("http://localhost:11434/v1", "qwen2.5", False),
    "vllm": ("http://localhost:8000/v1", "Qwen/Qwen2.5-7B-Instruct", False),
    "openrouter": ("https://openrouter.ai/api/v1", "meta-llama/llama-3.3-70b-instruct", True),
}

# Backends that hit a remote, rate-limited provider (must go through the limiter
# + usage tracker). Local servers and the on-device embedding model are exempt.
ONLINE_LLM_BACKENDS = {"nvidia", "google", "openrouter"}
ONLINE_EMBEDDING_BACKENDS = {"nvidia", "openai"}

# NVIDIA NeMo Retriever (nv-embedqa) embedding endpoints require an ``input_type``
# discriminator ("query" vs "passage"); plain OpenAI embeddings do not.
_NEEDS_INPUT_TYPE_PREFIXES = ("nvidia/", "nv-")


def _load_dotenv():
    """Minimal .env loader (no dependency): populate os.environ from ./.env.

    Existing environment variables win; only fills in unset keys.
    """
    for path in (".env", os.path.join("proprag_poc", ".env")):
        if not os.path.isfile(path):
            continue
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                k, v = k.strip(), v.strip().strip('"').strip("'")
                if k and k not in os.environ:
                    os.environ[k] = v


@dataclass
class POCConfig:
    """One and only configuration for the POC."""

    # ----- Paths -----
    data_dir: str = field(default="data")

    # ----- LLM endpoint -----
    # Default backend is the free NVIDIA-hosted API (OpenAI-compatible).
    llm_backend: Literal["nvidia", "google", "koboldcpp", "ollama", "vllm", "openrouter"] = "nvidia"
    llm_base_url: Optional[str] = None          # overrides preset if set
    llm_model: Optional[str] = None             # overrides preset if set
    llm_api_key: Optional[str] = None           # read from env / file if None
    temperature: float = 0.2                    # low temp for parseable JSON extraction
    max_completion_tokens: int = 4096
    seed: Optional[int] = None
    request_timeout: int = 600
    max_retry_attempts: int = 5
    # Concurrency for LLM calls. NVIDIA's free tier serializes requests server-side
    # anyway; concurrent workers just queue on their backend and inflate per-call
    # latency. Single worker keeps calls sequential, paced by rpm_limit.
    llm_max_workers: int = 1
    # gpt-oss emits a reasoning/analysis channel; strip it before JSON parsing.
    strip_reasoning: bool = True
    use_json_response_format: bool = True

    # ----- Rate limiting (chat LLM only, by default) -----
    # Free providers ban on overuse. One global sliding-60s-window limiter caps the
    # chat request rate. Only online backends are limited; cache hits are free.
    # NVIDIA's free tier is ~40 RPM (account-level, per model); 38 stays just under.
    # Embeddings run on the local ``sentence_transformers`` backend by default, so
    # they never touch this limiter (only the ``nvidia``/``openai`` embedding
    # backends would share it).
    rpm_limit: int = 38

    # ----- Embeddings (decoupled from the chat LLM) -----
    # Default: a local sentence-transformers model (no API, no rate limit, no
    # token cost) so every RAG system embeds with one model for a fair compare.
    # ``nvidia`` (NIM-hosted nv-embedqa) is kept as an online alternative.
    embedding_backend: Literal["nvidia", "sentence_transformers", "ollama", "openai"] = "sentence_transformers"
    embedding_model: str = "BAAI/bge-large-en-v1.5"
    embedding_base_url: Optional[str] = None    # for nvidia/ollama/openai backends
    embedding_api_key: Optional[str] = None
    embedding_batch_size: int = 32
    embedding_normalize: bool = True
    embedding_query_instruction: str = (
        "Represent this sentence for searching relevant passages: "
    )

    # ----- PDF ingestion / chunking -----
    chunk_max_tokens: int = 512
    chunk_overlap_tokens: int = 96
    tiktoken_encoding: str = "cl100k_base"
    drop_page_numbers: bool = True
    drop_repeated_headers: bool = True
    # A line repeated on >= this fraction of pages is treated as header/footer noise.
    header_repeat_fraction: float = 0.5

    # ----- Graph construction -----
    synonymy_sim_threshold: float = 0.8
    synonymy_top_k: int = 100
    is_directed_graph: bool = False

    # ----- Beam search -----
    beam_width: int = 4
    max_path_length: int = 3
    second_stage_filter_k: int = 40
    # "concatenate" re-embeds joined path text at query time -> fresh API calls per
    # path (slow + costly under an online, rate-limited embedder). The pooling modes
    # combine *precomputed* proposition embeddings with zero extra API calls, so they
    # are the sane default for an online backend. Keep concatenate only for local.
    embedding_combination: Literal[
        "concatenate", "average", "weighted_average", "max_pool", "attention"
    ] = "weighted_average"
    initial_proposition_seeds: int = 200

    # ----- Retrieval / PPR -----
    passage_node_weight: float = 0.05
    ppr_damping_stage1: float = 0.75
    ppr_damping_stage2: float = 0.45
    focus_top_k: int = 50
    select_top_k_paths: int = 20
    select_top_k_entities: int = 40
    retrieval_top_k: int = 10

    # ----- QA -----
    qa_top_k: int = 5
    history_max_turns: int = 6

    # ----- Indexing control -----
    force_index_from_scratch: bool = False
    force_extraction_from_scratch: bool = False

    def __post_init__(self):
        _load_dotenv()
        self._apply_env_overrides()

        base_url, model, needs_key = _LLM_PRESETS[self.llm_backend]
        if self.llm_base_url is None:
            self.llm_base_url = base_url
        if self.llm_model is None:
            self.llm_model = model
        if self.llm_api_key is None:
            self.llm_api_key = self._resolve_llm_key(needs_key)

        # Resolve the embedding endpoint for OpenAI-compatible / NVIDIA backends.
        if self.embedding_backend in ONLINE_EMBEDDING_BACKENDS:
            if self.embedding_base_url is None and self.embedding_backend == "nvidia":
                self.embedding_base_url = _LLM_PRESETS["nvidia"][0]
            if self.embedding_api_key is None:
                self.embedding_api_key = self._resolve_embedding_key()

        os.makedirs(self.data_dir, exist_ok=True)

    # -------------------------------------------------------------- env knobs
    def _apply_env_overrides(self):
        """Let the documented ``PROPRAG_*`` env vars (from .env) drive config.

        These were declared in ``.env.example`` but previously never read; wiring
        them here makes the .env file actually control the backend/model/rate limit.
        Env wins for these specific knobs (no caller in this repo sets them in code).
        """
        backend = os.environ.get("PROPRAG_LLM_BACKEND")
        if backend in _LLM_PRESETS:
            self.llm_backend = backend  # type: ignore[assignment]
        if os.environ.get("PROPRAG_LLM_MODEL"):
            self.llm_model = os.environ["PROPRAG_LLM_MODEL"]
        emb_backend = os.environ.get("PROPRAG_EMBEDDING_BACKEND")
        if emb_backend in ("nvidia", "sentence_transformers", "ollama", "openai"):
            self.embedding_backend = emb_backend  # type: ignore[assignment]
        if os.environ.get("PROPRAG_EMBEDDING_MODEL"):
            self.embedding_model = os.environ["PROPRAG_EMBEDDING_MODEL"]
        rpm = os.environ.get("PROPRAG_RPM_LIMIT")
        if rpm and rpm.strip().isdigit():
            self.rpm_limit = int(rpm.strip())

    # -------------------------------------------------------------- helpers
    @property
    def llm_is_online(self) -> bool:
        return self.llm_backend in ONLINE_LLM_BACKENDS

    @property
    def embedding_is_online(self) -> bool:
        return self.embedding_backend in ONLINE_EMBEDDING_BACKENDS

    @property
    def embedding_needs_input_type(self) -> bool:
        """nv-embedqa models require a query/passage ``input_type`` discriminator."""
        return self.embedding_backend == "nvidia" and self.embedding_model.startswith(
            _NEEDS_INPUT_TYPE_PREFIXES
        )

    def _resolve_llm_key(self, needs_key: bool) -> str:
        # Env var wins. NVIDIA -> NVIDIA_API_KEY; Gemini -> GEMINI/GOOGLE; else
        # PROPRAG_LLM_API_KEY / OPENAI_API_KEY. Then openrouter_api_key.txt; else dummy.
        env_names = []
        if self.llm_backend == "nvidia":
            env_names = ["NVIDIA_API_KEY", "PROPRAG_LLM_API_KEY"]
        elif self.llm_backend == "google":
            env_names = ["GEMINI_API_KEY", "GOOGLE_API_KEY", "PROPRAG_LLM_API_KEY"]
        else:
            env_names = ["PROPRAG_LLM_API_KEY", "OPENAI_API_KEY"]
        for name in env_names:
            key = os.environ.get(name)
            if key:
                return key
        if os.path.isfile("openrouter_api_key.txt"):
            with open("openrouter_api_key.txt", "r", encoding="utf-8") as f:
                txt = f.read().strip()
                if txt:
                    return txt
        if needs_key:
            hint = {
                "nvidia": "NVIDIA_API_KEY",
                "google": "GEMINI_API_KEY",
            }.get(self.llm_backend, "PROPRAG_LLM_API_KEY")
            raise ValueError(
                f"Backend '{self.llm_backend}' needs an API key. Set {hint} "
                "(e.g. in the .env file at the repo root)."
            )
        return "not-needed"

    def _resolve_embedding_key(self) -> str:
        # For the NVIDIA embedding backend reuse the NVIDIA / LLM key by default.
        for name in ("NVIDIA_API_KEY", "PROPRAG_EMBEDDING_API_KEY", "OPENAI_API_KEY"):
            key = os.environ.get(name)
            if key:
                return key
        if self.embedding_backend == "nvidia" and self.llm_backend == "nvidia" and self.llm_api_key:
            return self.llm_api_key
        if self.embedding_backend == "nvidia":
            raise ValueError(
                "Embedding backend 'nvidia' needs an API key. Set NVIDIA_API_KEY "
                "in the .env file at the repo root."
            )
        return "not-needed"


In [ ]:
%%writefile /content/proprag_poc/logging_setup.py
"""One-call logging setup so progress is visible in the terminal.

All package modules log under the ``proprag_poc`` logger tree, so configuring that
parent once routes NER/proposition extraction, embedding batches, LLM calls,
rate-limit waits and the compare run to stdout. Streamlit forwards stdout to the
terminal it was launched from.
"""

from __future__ import annotations

import logging
import os
import sys

_CONFIGURED = False


def setup_logging(level: int | None = None) -> None:
    """Attach a stdout handler to the ``proprag_poc`` logger (idempotent).

    Level comes from the ``PROPRAG_LOG_LEVEL`` env var (default INFO).
    """
    global _CONFIGURED
    if _CONFIGURED:
        return
    if level is None:
        level = getattr(logging, os.environ.get("PROPRAG_LOG_LEVEL", "INFO").upper(), logging.INFO)
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-5s | %(name)s | %(message)s", "%H:%M:%S")
    )
    root = logging.getLogger("proprag_poc")
    root.setLevel(level)
    root.handlers.clear()
    root.addHandler(handler)
    root.propagate = False
    _CONFIGURED = True


In [ ]:
%%writefile /content/proprag_poc/core/__init__.py


In [ ]:
%%writefile /content/proprag_poc/core/ids.py
"""Stable content-hash node identifiers (ported from reference ``compute_mdhash_id``)."""

from __future__ import annotations

import hashlib


def compute_mdhash_id(content: str, prefix: str = "") -> str:
    """Return ``{prefix}{md5(content)}``.

    Used for every graph node id (entity-, chunk-, proposition-). Never use raw
    strings as node keys.
    """
    return prefix + hashlib.md5(content.encode("utf-8")).hexdigest()


In [ ]:
%%writefile /content/proprag_poc/core/metrics.py
"""Usage / cost / latency instrumentation shared across RAG systems.

A process-wide ``UsageTracker`` records every chat and embedding call (token
counts, latency, cache hits). Query-time calls are attributed to a *scope* (one
per RAG system) so the Compare view can show token consumption, call counts and
latency side-by-side. Index-time calls land in the default scope and are reported
once (the corpus/stores/graph are shared by all three systems).

The tracker is a singleton so the chat client and embedding client can report
into it without threading an object through every constructor.
"""

from __future__ import annotations

import threading
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Dict, List, Optional

_DEFAULT_SCOPE = "index"


@dataclass
class CallRecord:
    kind: str            # "chat" | "embed"
    model: str
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_s: float = 0.0
    rate_wait_s: float = 0.0
    cache_hit: bool = False
    batch_size: int = 1  # number of texts in an embedding request


@dataclass
class Snapshot:
    """Aggregated usage for one scope."""

    scope: str
    chat_calls: int = 0
    chat_cache_hits: int = 0
    chat_prompt_tokens: int = 0
    chat_completion_tokens: int = 0
    embed_calls: int = 0
    embed_cache_hits: int = 0
    embed_texts: int = 0
    embed_tokens: int = 0
    chat_latency_s: float = 0.0
    embed_latency_s: float = 0.0
    rate_wait_s: float = 0.0

    @property
    def chat_total_tokens(self) -> int:
        return self.chat_prompt_tokens + self.chat_completion_tokens

    @property
    def total_tokens(self) -> int:
        return self.chat_total_tokens + self.embed_tokens

    @property
    def total_calls(self) -> int:
        return self.chat_calls + self.embed_calls

    def as_dict(self) -> Dict[str, float]:
        return {
            "chat_calls": self.chat_calls,
            "chat_cache_hits": self.chat_cache_hits,
            "chat_prompt_tokens": self.chat_prompt_tokens,
            "chat_completion_tokens": self.chat_completion_tokens,
            "chat_total_tokens": self.chat_total_tokens,
            "embed_calls": self.embed_calls,
            "embed_cache_hits": self.embed_cache_hits,
            "embed_texts": self.embed_texts,
            "embed_tokens": self.embed_tokens,
            "total_tokens": self.total_tokens,
            "total_calls": self.total_calls,
            "chat_latency_s": round(self.chat_latency_s, 3),
            "embed_latency_s": round(self.embed_latency_s, 3),
            "rate_wait_s": round(self.rate_wait_s, 3),
        }


class UsageTracker:
    def __init__(self):
        self._lock = threading.Lock()
        # scope name -> list of CallRecord. Thread-local current scope.
        self._records: Dict[str, List[CallRecord]] = {}
        self._tl = threading.local()

    # ------------------------------------------------------------- scoping
    def _current_scope(self) -> str:
        return getattr(self._tl, "scope", _DEFAULT_SCOPE)

    @contextmanager
    def scope(self, name: str):
        """Attribute calls made on this thread to ``name`` for the block's duration."""
        prev = getattr(self._tl, "scope", None)
        self._tl.scope = name
        try:
            yield
        finally:
            if prev is None:
                del self._tl.scope
            else:
                self._tl.scope = prev

    # ------------------------------------------------------------- record
    def record(self, rec: CallRecord, scope: Optional[str] = None):
        scope = scope or self._current_scope()
        with self._lock:
            self._records.setdefault(scope, []).append(rec)

    # ------------------------------------------------------------- read
    def snapshot(self, scope: str) -> Snapshot:
        with self._lock:
            recs = list(self._records.get(scope, []))
        snap = Snapshot(scope=scope)
        for r in recs:
            if r.kind == "chat":
                snap.chat_calls += 1
                snap.chat_cache_hits += int(r.cache_hit)
                snap.chat_prompt_tokens += r.prompt_tokens
                snap.chat_completion_tokens += r.completion_tokens
                snap.chat_latency_s += r.latency_s
            else:
                snap.embed_calls += 1
                snap.embed_cache_hits += int(r.cache_hit)
                snap.embed_texts += r.batch_size
                snap.embed_tokens += r.prompt_tokens
                snap.embed_latency_s += r.latency_s
            snap.rate_wait_s += r.rate_wait_s
        return snap

    def reset(self, scope: Optional[str] = None):
        with self._lock:
            if scope is None:
                self._records.clear()
            else:
                self._records.pop(scope, None)

    def scopes(self) -> List[str]:
        with self._lock:
            return list(self._records.keys())


# ----------------------------------------------------------------- singleton
_TRACKER: Optional[UsageTracker] = None
_TRACKER_LOCK = threading.Lock()


def get_usage_tracker() -> UsageTracker:
    global _TRACKER
    with _TRACKER_LOCK:
        if _TRACKER is None:
            _TRACKER = UsageTracker()
        return _TRACKER


In [ ]:
%%writefile /content/proprag_poc/core/store.py
"""Parquet-backed text store + aligned vector store (slim port of the reference
EmbeddingStore). One store per namespace: chunk / entity / proposition.
"""

from __future__ import annotations

import os
from typing import Dict, List

import numpy as np
import pandas as pd

from ..embedding.encoder import EmbeddingModel
from .ids import compute_mdhash_id


class EmbeddingStore:
    def __init__(self, embedding_model: EmbeddingModel, directory: str, namespace: str):
        self.embedding_model = embedding_model
        self.namespace = namespace
        self.directory = directory
        os.makedirs(directory, exist_ok=True)
        self._text_path = os.path.join(directory, f"{namespace}_text.parquet")
        self._vec_path = os.path.join(directory, f"{namespace}_vectors.npy")
        self._ids: List[str] = []
        self._content: List[str] = []
        self._id_to_row: Dict[str, int] = {}
        self._vectors: np.ndarray | None = None
        self._load()

    # ----------------------------------------------------------------- io
    def _load(self):
        if os.path.isfile(self._text_path):
            df = pd.read_parquet(self._text_path)
            self._ids = df["hash_id"].tolist()
            self._content = df["content"].tolist()
            self._id_to_row = {h: i for i, h in enumerate(self._ids)}
        if os.path.isfile(self._vec_path):
            self._vectors = np.load(self._vec_path)

    def _persist(self):
        pd.DataFrame({"hash_id": self._ids, "content": self._content}).to_parquet(
            self._text_path, index=False
        )
        if self._vectors is not None:
            np.save(self._vec_path, self._vectors)

    # ------------------------------------------------------------- mutation
    def insert_strings(self, texts: List[str]):
        """Hash, dedupe against existing, encode new, append. Idempotent."""
        new_ids, new_texts = [], []
        seen = set()
        for t in texts:
            hid = compute_mdhash_id(t, prefix=f"{self.namespace}-")
            if hid in self._id_to_row or hid in seen:
                continue
            seen.add(hid)
            new_ids.append(hid)
            new_texts.append(t)
        if not new_ids:
            return
        new_vecs = self.embedding_model.batch_encode(new_texts, norm=True)
        start = len(self._ids)
        self._ids.extend(new_ids)
        self._content.extend(new_texts)
        for j, hid in enumerate(new_ids):
            self._id_to_row[hid] = start + j
        self._vectors = (
            new_vecs if self._vectors is None else np.vstack([self._vectors, new_vecs])
        )
        self._persist()

    # -------------------------------------------------------------- queries
    def get_row(self, hash_id: str) -> Dict[str, str] | None:
        idx = self._id_to_row.get(hash_id)
        return None if idx is None else {"content": self._content[idx]}

    def get_embedding(self, hash_id: str) -> np.ndarray:
        return self._vectors[self._id_to_row[hash_id]]

    def get_embeddings(self, hash_ids: List[str]) -> np.ndarray:
        if not hash_ids:
            return np.zeros((0, self._vectors.shape[1]), dtype=np.float32)
        return np.stack([self._vectors[self._id_to_row[h]] for h in hash_ids])

    def get_all_ids(self) -> List[str]:
        return list(self._ids)

    def get_text_for_all_rows(self) -> Dict[str, Dict[str, str]]:
        return {h: {"content": self._content[i]} for i, h in enumerate(self._ids)}

    def __len__(self):
        return len(self._ids)


In [ ]:
%%writefile /content/proprag_poc/core/extraction.py
"""NER + proposition extraction (ported from reference EnhancedOpenIE +
PropositionExtractor). Parallel LLM calls, constrained propositions, and a
multi-tier JSON-repair fallback chain.
"""

from __future__ import annotations

import json
import logging
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List

from ..llm.client import LLMClient
from ..llm import prompts

logger = logging.getLogger(__name__)

# Matches {"text": "...", "entities": [ ... ]} proposition objects.
_PROP_OBJ = re.compile(
    r'\{\s*"text"\s*:\s*"(?P<text>(?:\\.|[^"\\])*)"\s*,\s*"entities"\s*:\s*'
    r'\[(?P<ents>[^\]]*)\]\s*\}',
    re.DOTALL,
)
_STR = re.compile(r'"((?:\\.|[^"\\])*)"')


def _loads_lenient(raw: str):
    """Best-effort parse; strips code fences before json.loads."""
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    return json.loads(s)


def _parse_entities(raw: str) -> List[str]:
    try:
        obj = _loads_lenient(raw)
        ents = obj.get("entities", []) if isinstance(obj, dict) else []
        return [str(e) for e in ents if str(e).strip()]
    except Exception:
        # Fallback: pull the entities array via regex.
        m = re.search(r'"entities"\s*:\s*\[(.*?)\]', raw, re.DOTALL)
        if not m:
            return []
        return [s.strip() for s in _STR.findall(m.group(1)) if s.strip()]


def _parse_propositions(raw: str) -> List[Dict]:
    """Parse proposition objects, tolerating malformed surrounding JSON."""
    try:
        obj = _loads_lenient(raw)
        props = obj.get("propositions", []) if isinstance(obj, dict) else []
        out = [
            {"text": p["text"], "entities": list(p.get("entities", []))}
            for p in props
            if isinstance(p, dict) and p.get("text")
        ]
        if out:
            return out
    except Exception:
        pass
    # Regex fallback: extract each {"text","entities"} object independently.
    out = []
    for m in _PROP_OBJ.finditer(raw):
        ents = [s for s in _STR.findall(m.group("ents"))]
        out.append({"text": m.group("text"), "entities": ents})
    return out


class Extractor:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    # ------------------------------------------------------------------- NER
    def ner(self, passage: str) -> List[str]:
        raw, _, _ = self.llm.infer(prompts.ner_messages(passage), json_mode=True)
        entities = _parse_entities(raw)
        if not entities:
            fixed, _, _ = self.llm.infer(prompts.fix_json_messages(raw), json_mode=True)
            entities = _parse_entities(fixed)
        # Dedupe, keep order.
        return list(dict.fromkeys(entities))

    # --------------------------------------------------------- propositions
    def propositions(self, passage: str, named_entities: List[str]) -> List[Dict]:
        msgs = prompts.proposition_messages(passage, json.dumps(named_entities))
        raw, _, _ = self.llm.infer(msgs, json_mode=True)
        props = _parse_propositions(raw)
        if not props:
            fixed, _, _ = self.llm.infer(prompts.fix_json_messages(raw), json_mode=True)
            props = _parse_propositions(fixed)
        return self._constrain(props, named_entities)

    def _constrain(self, props: List[Dict], allowed: List[str]) -> List[Dict]:
        """Drop entities not in the NER set (hard constraint from the reference)."""
        allowed_set = set(allowed)
        cleaned = []
        for p in props:
            ents = [e for e in p["entities"] if e in allowed_set]
            text = (p.get("text") or "").strip()
            if text:
                cleaned.append({"text": text, "entities": ents})
        return cleaned

    # --------------------------------------------------------------- batch
    def batch_extract(self, chunk_texts: Dict[str, str]) -> Dict[str, List[Dict]]:
        """chunk_id -> [{text, entities}]. Parallel NER then parallel propositions."""
        keys = list(chunk_texts.keys())
        n = len(keys)
        max_workers = self.llm.config.llm_max_workers
        logger.info("extraction: %d chunks (NER then propositions), %d workers", n, max_workers)

        ner_results: Dict[str, List[str]] = {}
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futs = {ex.submit(self.ner, chunk_texts[k]): k for k in keys}
            for done, fut in enumerate(as_completed(futs), 1):
                k = futs[fut]
                ner_results[k] = fut.result()
                logger.info("NER %d/%d chunks done", done, n)

        prop_results: Dict[str, List[Dict]] = {}
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futs = {ex.submit(self.propositions, chunk_texts[k], ner_results[k]): k for k in keys}
            for done, fut in enumerate(as_completed(futs), 1):
                k = futs[fut]
                prop_results[k] = fut.result()
                logger.info("propositions %d/%d chunks done", done, n)
        logger.info("extraction complete: %d chunks", n)
        return prop_results


In [ ]:
%%writefile /content/proprag_poc/core/graph_builder.py
"""Build the undirected PropRAG knowledge graph with igraph.

Node types: entity-<md5>, chunk-<md5> (propositions are NOT graph nodes; they
live in the embedding store + maps). Edge weights:
  - entity-entity co-occurrence: INTEGER count (entities sharing a proposition).
  - entity-passage: 1.0.
  - entity-entity synonymy: FLOAT similarity in [threshold, 1).
The integer-vs-float distinction is how beam search later identifies synonymy
edges, so the invariant must hold.
"""

from __future__ import annotations

import logging
from collections import defaultdict
from typing import Dict, List, Tuple

import igraph as ig
import numpy as np

from ..config import POCConfig
from .ids import compute_mdhash_id
from .store import EmbeddingStore

logger = logging.getLogger(__name__)


class GraphBuilder:
    def __init__(self, config: POCConfig):
        self.config = config

    def build(
        self,
        chunk_ids: List[str],
        chunk_propositions: Dict[str, List[Dict]],
        entity_store: EmbeddingStore,
        chunk_store: EmbeddingStore,
    ) -> ig.Graph:
        stats: Dict[Tuple[str, str], float] = {}

        # 1) entity-entity co-occurrence (fully connect entities within a prop).
        for chunk_id in chunk_ids:
            for prop in chunk_propositions.get(chunk_id, []):
                ekeys = [
                    compute_mdhash_id(e, prefix="entity-") for e in prop["entities"]
                ]
                for i in range(len(ekeys)):
                    for j in range(i + 1, len(ekeys)):
                        a, b = sorted((ekeys[i], ekeys[j]))
                        if a == b:
                            continue
                        stats[(a, b)] = stats.get((a, b), 0.0) + 1.0

        # 2) entity-passage edges.
        for chunk_id in chunk_ids:
            ents = set()
            for prop in chunk_propositions.get(chunk_id, []):
                for e in prop["entities"]:
                    ents.add(compute_mdhash_id(e, prefix="entity-"))
            for ekey in ents:
                stats[(chunk_id, ekey)] = 1.0

        # 3) entity-entity synonymy (KNN, float weights; only where no edge yet).
        self._add_synonymy(stats, entity_store)

        return self._assemble(stats, entity_store, chunk_store)

    # ------------------------------------------------------------- synonymy
    def _add_synonymy(self, stats, entity_store: EmbeddingStore):
        ids = entity_store.get_all_ids()
        if len(ids) < 2:
            return
        embs = entity_store.get_embeddings(ids)  # normalized
        sims = embs @ embs.T
        thr = self.config.synonymy_sim_threshold
        topk = self.config.synonymy_top_k
        np.fill_diagonal(sims, -1.0)
        for i, id_i in enumerate(ids):
            row = sims[i]
            cand = np.argsort(row)[::-1][:topk]
            for j in cand:
                score = float(row[j])
                if score < thr:
                    break
                a, b = sorted((id_i, ids[j]))
                if (a, b) in stats:  # keep integer co-occurrence invariant
                    continue
                # Ensure strictly non-integer so beam search reads it as synonymy.
                stats[(a, b)] = min(0.999999, max(thr, score))

    # ------------------------------------------------------------- assemble
    def _assemble(self, stats, entity_store, chunk_store) -> ig.Graph:
        names = list(entity_store.get_all_ids()) + list(chunk_store.get_all_ids())
        name_to_idx = {n: i for i, n in enumerate(names)}
        g = ig.Graph(directed=self.config.is_directed_graph)
        g.add_vertices(len(names))
        g.vs["name"] = names

        edges, weights = [], []
        for (a, b), w in stats.items():
            if a in name_to_idx and b in name_to_idx and a != b:
                edges.append((name_to_idx[a], name_to_idx[b]))
                weights.append(float(w))
        g.add_edges(edges)
        g.es["weight"] = weights
        logger.info("Graph built: %d nodes, %d edges", g.vcount(), g.ecount())
        return g


In [ ]:
%%writefile /content/proprag_poc/core/retriever.py
"""Two-stage, LLM-free retrieval (ported from the reference
``graph_search_with_proposition_entities`` + ``run_ppr``):

DPR passage prior -> beam search -> path entities seed PPR #1 (full graph,
d=0.75) -> focused subgraph of top passages -> beam #2 -> PPR #2 (d=0.45) ->
merged passage ranking. Falls back to dense passage retrieval if graph signal
is absent.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Dict, List

import numpy as np

from ..config import POCConfig
from ..embedding.encoder import EmbeddingModel
from .beam_search import BeamSearchPathFinder
from .ids import compute_mdhash_id

logger = logging.getLogger(__name__)


def _min_max(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


@dataclass
class RetrievedPassage:
    chunk_id: str
    text: str
    score: float


class Retriever:
    system_name = "PropRAG"

    def __init__(self, corpus, embedding_model: EmbeddingModel, config: POCConfig):
        self.config = config
        self.embedding_model = embedding_model
        self.chunk_embedding_store = corpus.chunk_store
        self.entity_embedding_store = corpus.entity_store
        self.proposition_embedding_store = corpus.proposition_store
        self.graph = corpus.graph
        self.proposition_to_entities_map = corpus.proposition_to_entities_map
        self.chunk_propositions = corpus.chunk_propositions
        self._prepare()

    # ----------------------------------------------------------------- setup
    def _prepare(self):
        self.entity_node_keys = self.entity_embedding_store.get_all_ids()
        self.passage_node_keys = self.chunk_embedding_store.get_all_ids()
        self.proposition_node_keys = self.proposition_embedding_store.get_all_ids()

        self.node_name_to_vertex_idx = {v["name"]: i for i, v in enumerate(self.graph.vs)}
        self.passage_node_idxs = [
            self.node_name_to_vertex_idx[k]
            for k in self.passage_node_keys
            if k in self.node_name_to_vertex_idx
        ]
        self.passage_key_to_row = {k: i for i, k in enumerate(self.passage_node_keys)}

        self.passage_embeddings = self.chunk_embedding_store.get_embeddings(self.passage_node_keys)
        prop_embs = self.proposition_embedding_store.get_embeddings(self.proposition_node_keys)
        self.prop_key_to_propositions = {
            k: prop_embs[i] for i, k in enumerate(self.proposition_node_keys)
        }
        self.all_proposition_embeddings = prop_embs

        self.beam_search = BeamSearchPathFinder(
            self,
            beam_width=self.config.beam_width,
            max_path_length=self.config.max_path_length,
            embedding_combination=self.config.embedding_combination,
            second_stage_filter_k=self.config.second_stage_filter_k,
        )

    # --------------------------------------------------------- dense passages
    def _dpr_scores(self, query: str) -> np.ndarray:
        q = self.embedding_model.batch_encode(
            query, instruction=self.config.embedding_query_instruction, norm=True,
            input_type="query",
        )
        scores = (self.passage_embeddings @ q.squeeze().T).squeeze()
        return _min_max(np.atleast_1d(scores))  # aligned to passage_node_keys

    # --------------------------------------------------------------- PPR
    def _run_ppr(self, graph, reset_prob, damping):
        reset_prob = np.where(np.isnan(reset_prob) | (reset_prob < 0), 0, reset_prob)
        if reset_prob.sum() == 0:
            return None
        return np.array(
            graph.personalized_pagerank(
                vertices=range(graph.vcount()),
                damping=damping,
                directed=False,
                weights="weight",
                reset=list(reset_prob),
                implementation="prpack",
            )
        )

    # --------------------------------------------------------------- retrieve
    def retrieve(self, query: str, top_k: int = None) -> List[RetrievedPassage]:
        top_k = top_k or self.config.retrieval_top_k
        n_vert = self.graph.vcount()
        dpr = self._dpr_scores(query)

        # Stage 1: beam over full graph -> entity seeds.
        paths = self.beam_search.find_paths(query)[: self.config.select_top_k_paths]
        top_entities = self.beam_search.get_entities_from_paths(paths)[
            : self.config.select_top_k_entities
        ]

        phrase_weights = np.zeros(n_vert)
        for ekey, _ in top_entities:
            idx = self.node_name_to_vertex_idx.get(ekey)
            if idx is not None:
                phrase_weights[idx] = 1.0
        phrase_weights = _min_max(phrase_weights)

        passage_weights = np.zeros(n_vert)
        for row, pkey in enumerate(self.passage_node_keys):
            idx = self.node_name_to_vertex_idx.get(pkey)
            if idx is not None:
                passage_weights[idx] = dpr[row] * self.config.passage_node_weight

        node_weights = phrase_weights + passage_weights
        ppr1 = self._run_ppr(self.graph, node_weights, self.config.ppr_damping_stage1)
        if ppr1 is None:
            return self._dpr_fallback(dpr, top_k)

        doc_scores = np.array([ppr1[i] for i in self.passage_node_idxs])
        final = self._focused_stage(query, doc_scores, dpr)
        order = np.argsort(final)[::-1][:top_k]
        return [
            RetrievedPassage(
                chunk_id=self.passage_node_keys[i],
                text=self.chunk_embedding_store.get_row(self.passage_node_keys[i])["content"],
                score=float(final[i]),
            )
            for i in order
        ]

    # ---------------------------------------------------- focused 2nd stage
    def _focused_stage(self, query, first_doc_scores, dpr):
        focus = self.config.focus_top_k
        top_doc_rows = np.argsort(first_doc_scores)[::-1][:focus]
        top_doc_keys = [self.passage_node_keys[r] for r in top_doc_rows]
        valid = [k for k in top_doc_keys if k in self.node_name_to_vertex_idx]
        if not valid:
            return first_doc_scores

        # Propositions belonging to the focused passages.
        focus_props = []
        for k in valid:
            for prop in self.chunk_propositions.get(k, []):
                focus_props.append(compute_mdhash_id(prop["text"], prefix="proposition-"))
        focus_props = [p for p in focus_props if p in self.prop_key_to_propositions]
        if not focus_props:
            return first_doc_scores

        # Subgraph: focused passages + their entity neighbors.
        doc_vertices = [self.node_name_to_vertex_idx[k] for k in valid]
        include = set(doc_vertices)
        for dv in doc_vertices:
            for nb in self.graph.neighbors(dv, mode="all"):
                if self.graph.vs[nb]["name"].startswith("entity-"):
                    include.add(nb)
        sub = self.graph.induced_subgraph(list(include))
        if sub.vcount() == 0:
            return first_doc_scores
        sub_name_to_idx = {v["name"]: i for i, v in enumerate(sub.vs)}

        # Re-target beam search onto the subgraph + focused proposition set.
        bs = self.beam_search
        orig_graph, orig_map = bs.active_graph, bs.node_name_to_vertex_idx
        bs.active_graph = sub
        bs.set_node_name_to_vertex_idx(sub_name_to_idx)
        bs.clear_caches()
        try:
            paths2 = bs.find_paths(query, prop_set=focus_props)[:5]
            ents2 = bs.get_entities_from_paths(paths2)[:5]
        finally:
            bs.active_graph = orig_graph
            bs.set_node_name_to_vertex_idx(orig_map)
            bs.clear_caches()

        sub_phrase = np.zeros(sub.vcount())
        for ekey, scores in ents2:
            si = sub_name_to_idx.get(ekey)
            if si is not None:
                sub_phrase[si] = max(scores) if scores else 0.0
        if sub_phrase.sum() > 0:
            sub_phrase = _min_max(sub_phrase)

        sub_passage = np.zeros(sub.vcount())
        for row, pkey in enumerate(self.passage_node_keys):
            si = sub_name_to_idx.get(pkey)
            if si is not None:
                sub_passage[si] = dpr[row] * self.config.passage_node_weight

        sub_weights = sub_phrase + sub_passage
        sub_ppr = self._run_ppr(sub, sub_weights, self.config.ppr_damping_stage2)
        if sub_ppr is None:
            return first_doc_scores

        # Merge: focused PPR overrides scores for passages in the subgraph;
        # scale the rest below the focused floor.
        floor = float(np.min(sub_ppr)) * 0.5
        final = first_doc_scores * floor
        for si, score in enumerate(sub_ppr):
            name = sub.vs[si]["name"]
            if name.startswith("chunk-") and name in self.passage_key_to_row:
                final[self.passage_key_to_row[name]] = score
        return final

    def _dpr_fallback(self, dpr, top_k):
        order = np.argsort(dpr)[::-1][:top_k]
        return [
            RetrievedPassage(
                chunk_id=self.passage_node_keys[i],
                text=self.chunk_embedding_store.get_row(self.passage_node_keys[i])["content"],
                score=float(dpr[i]),
            )
            for i in order
        ]


In [ ]:
%%writefile /content/proprag_poc/core/baseline_retrievers.py
"""Baseline retrievers for head-to-head comparison against PropRAG.

Both reuse the *same* embedding stores and knowledge graph that PropRAG builds, so
the only differences measured are retrieval strategy and its cost — embeddings,
chunks and the graph are shared and built once.

  * ``BaseRAGRetriever`` — classic dense passage retrieval: embed the query, take
    the top-k passages by cosine similarity. No graph, no LLM at query time.

  * ``GraphRAGRetriever`` — a lightweight, summary-free GraphRAG: run NER on the
    query (one LLM call), match the named entities to entity nodes, expand one hop
    to neighbouring entities, gather the passages attached to that entity set, and
    rank them by (graph overlap with query entities) + (dense similarity). Falls
    back to dense retrieval when no query entity matches the graph.
"""

from __future__ import annotations

import logging
from typing import List

import numpy as np

from ..config import POCConfig
from ..embedding.encoder import EmbeddingModel
from .extraction import Extractor
from .ids import compute_mdhash_id
from .retriever import RetrievedPassage, _min_max

logger = logging.getLogger(__name__)


class BaseRAGRetriever:
    system_name = "BaseRAG"

    def __init__(self, corpus, embedding_model: EmbeddingModel, config: POCConfig):
        self.config = config
        self.embedding_model = embedding_model
        self.chunk_store = corpus.chunk_store
        self.passage_keys = self.chunk_store.get_all_ids()
        self.passage_embeddings = self.chunk_store.get_embeddings(self.passage_keys)

    def retrieve(self, query: str, top_k: int = None) -> List[RetrievedPassage]:
        top_k = top_k or self.config.retrieval_top_k
        q = self.embedding_model.batch_encode(
            query, instruction=self.config.embedding_query_instruction, norm=True,
            input_type="query",
        )
        scores = np.atleast_1d((self.passage_embeddings @ q.squeeze().T).squeeze())
        order = np.argsort(scores)[::-1][:top_k]
        return [
            RetrievedPassage(
                chunk_id=self.passage_keys[i],
                text=self.chunk_store.get_row(self.passage_keys[i])["content"],
                score=float(scores[i]),
            )
            for i in order
        ]


class GraphRAGRetriever:
    system_name = "GraphRAG"

    # Cosine floor for matching a query NER phrase to an entity node by embedding.
    _ENTITY_MATCH_THRESHOLD = 0.6

    def __init__(self, corpus, embedding_model: EmbeddingModel, config: POCConfig,
                 extractor: Extractor):
        self.config = config
        self.embedding_model = embedding_model
        self.extractor = extractor
        self.graph = corpus.graph
        self.chunk_store = corpus.chunk_store
        self.entity_store = corpus.entity_store

        self.passage_keys = self.chunk_store.get_all_ids()
        self.passage_key_to_row = {k: i for i, k in enumerate(self.passage_keys)}
        self.passage_embeddings = self.chunk_store.get_embeddings(self.passage_keys)

        self.entity_keys = self.entity_store.get_all_ids()
        self.entity_embeddings = (
            self.entity_store.get_embeddings(self.entity_keys)
            if self.entity_keys else np.zeros((0, 1), dtype=np.float32)
        )
        self.name_to_idx = {v["name"]: i for i, v in enumerate(self.graph.vs)}

    # -------------------------------------------------------------- helpers
    def _dense_scores(self, query: str) -> np.ndarray:
        q = self.embedding_model.batch_encode(
            query, instruction=self.config.embedding_query_instruction, norm=True,
            input_type="query",
        )
        return _min_max(np.atleast_1d((self.passage_embeddings @ q.squeeze().T).squeeze()))

    def _match_query_entities(self, phrases: List[str]) -> List[str]:
        """Map query NER phrases to entity node keys (exact id, then embedding NN)."""
        matched, unresolved = set(), []
        for p in phrases:
            key = compute_mdhash_id(p.strip(), prefix="entity-")
            if key in self.name_to_idx:
                matched.add(key)
            else:
                unresolved.append(p)
        if unresolved and len(self.entity_keys) > 0:
            pe = self.embedding_model.batch_encode(
                unresolved, norm=True, input_type="query"
            )
            sims = pe @ self.entity_embeddings.T  # (n_phrases, n_entities)
            for row in sims:
                j = int(np.argmax(row))
                if float(row[j]) >= self._ENTITY_MATCH_THRESHOLD:
                    matched.add(self.entity_keys[j])
        return list(matched)

    # -------------------------------------------------------------- retrieve
    def retrieve(self, query: str, top_k: int = None) -> List[RetrievedPassage]:
        top_k = top_k or self.config.retrieval_top_k
        dense = self._dense_scores(query)

        try:
            phrases = self.extractor.ner(query)
        except Exception as e:  # noqa: BLE001 - degrade to dense on NER failure
            logger.warning("GraphRAG query NER failed, using dense: %s", e)
            phrases = []
        seeds = self._match_query_entities(phrases) if phrases else []

        if not seeds:
            return self._dense_top_k(dense, top_k)

        # Expand: seed entities + one hop of neighbouring entities.
        entity_set = set(seeds)
        for ekey in seeds:
            vi = self.name_to_idx.get(ekey)
            if vi is None:
                continue
            for nb in self.graph.neighbors(vi, mode="all"):
                name = self.graph.vs[nb]["name"]
                if name.startswith("entity-"):
                    entity_set.add(name)

        # Passages attached to that entity set, with an overlap count vs the seeds.
        overlap = np.zeros(len(self.passage_keys))
        for ekey in entity_set:
            vi = self.name_to_idx.get(ekey)
            if vi is None:
                continue
            weight = 1.0 if ekey in seeds else 0.4  # seeds count more than hop-neighbours
            for nb in self.graph.neighbors(vi, mode="all"):
                name = self.graph.vs[nb]["name"]
                row = self.passage_key_to_row.get(name)
                if row is not None:
                    overlap[row] += weight

        if overlap.sum() == 0:
            return self._dense_top_k(dense, top_k)

        # Final score: graph overlap (primary) blended with dense similarity.
        final = _min_max(overlap) + 0.5 * dense
        order = np.argsort(final)[::-1][:top_k]
        return [
            RetrievedPassage(
                chunk_id=self.passage_keys[i],
                text=self.chunk_store.get_row(self.passage_keys[i])["content"],
                score=float(final[i]),
            )
            for i in order
            if final[i] > 0
        ]

    def _dense_top_k(self, dense: np.ndarray, top_k: int) -> List[RetrievedPassage]:
        order = np.argsort(dense)[::-1][:top_k]
        return [
            RetrievedPassage(
                chunk_id=self.passage_keys[i],
                text=self.chunk_store.get_row(self.passage_keys[i])["content"],
                score=float(dense[i]),
            )
            for i in order
        ]


In [ ]:
%%writefile /content/proprag_poc/core/beam_search.py
"""LLM-free beam search over proposition paths (ported from the reference
BeamSearchPathFinder; the trained-predictor combination mode is removed).

Paths grow by hopping to propositions that share an entity (or a synonymous
entity) with the path's last proposition. Each path is scored by a combined
embedding of its propositions against the query.
"""

from __future__ import annotations

import copy
import logging
from collections import defaultdict
from typing import Any, Dict, List, Tuple

import numpy as np

from .ids import compute_mdhash_id

logger = logging.getLogger(__name__)


class BeamSearchPathFinder:
    def __init__(self, rag, beam_width=4, max_path_length=3,
                 embedding_combination="concatenate", sim_threshold=0.75,
                 second_stage_filter_k=0):
        self.rag = rag
        self.beam_width = beam_width
        self.max_path_length = max_path_length
        self.embedding_combination = embedding_combination
        self.sim_threshold = sim_threshold
        self.second_stage_filter_k = second_stage_filter_k
        self.active_graph = rag.graph
        self.node_name_to_vertex_idx = rag.node_name_to_vertex_idx

        # Inverse map: entity-key -> [proposition-key].
        self.entity_to_propositions_map = defaultdict(list)
        for prop_key, entities in rag.proposition_to_entities_map.items():
            for entity in entities:
                ekey = compute_mdhash_id(entity, prefix="entity-")
                self.entity_to_propositions_map[ekey].append(prop_key)
        self.clear_caches()

    # ----------------------------------------------------------------- setup
    def set_node_name_to_vertex_idx(self, mapping):
        self.node_name_to_vertex_idx = mapping

    def clear_caches(self):
        self.synonymous_entities_cache = {}
        self.connected_propositions_cache = {}

    # --------------------------------------------------------------- lookups
    def get_proposition_text(self, prop_key: str) -> str:
        row = self.rag.proposition_embedding_store.get_row(prop_key)
        return row["content"] if row else ""

    def get_entity_text(self, entity_key: str) -> str:
        row = self.rag.entity_embedding_store.get_row(entity_key)
        return row["content"] if row else ""

    def get_proposition_embeddings(self, prop_keys: List[str]) -> np.ndarray:
        return np.array([self.rag.prop_key_to_propositions[k] for k in prop_keys])

    def find_entities_in_proposition(self, prop_key: str) -> List[str]:
        ents = self.rag.proposition_to_entities_map.get(prop_key, [])
        return [compute_mdhash_id(e, prefix="entity-") for e in ents]

    def find_synonymous_entities(self, entity_key: str) -> List[Tuple[str, float]]:
        if entity_key in self.synonymous_entities_cache:
            return self.synonymous_entities_cache[entity_key]
        synonyms = []
        idx = self.node_name_to_vertex_idx.get(entity_key)
        if idx is not None:
            try:
                for nb in self.active_graph.neighbors(idx, mode="all"):
                    name = self.active_graph.vs[nb]["name"]
                    if not name.startswith("entity-"):
                        continue
                    eid = self.active_graph.get_eid(idx, nb, error=False)
                    if eid == -1:
                        continue
                    w = self.active_graph.es[eid]["weight"]
                    if isinstance(w, float) and not float(w).is_integer() and w >= self.sim_threshold:
                        synonyms.append((name, float(w)))
            except Exception as e:  # noqa: BLE001
                logger.debug("synonym lookup failed: %s", e)
        self.synonymous_entities_cache[entity_key] = synonyms
        return synonyms

    def find_connected_propositions(self, prop_key: str, prop_set=None):
        if prop_key in self.connected_propositions_cache:
            cached, processed = self.connected_propositions_cache[prop_key]
            return cached, processed
        connected, processed = [], set()
        for ekey in self.find_entities_in_proposition(prop_key):
            for other in self.entity_to_propositions_map.get(ekey, []):
                if other != prop_key and other not in processed and (
                    prop_set is None or other in prop_set
                ):
                    connected.append((other, [{"type": "exact", "entity1": ekey,
                                               "entity2": ekey, "similarity": 1.0}]))
                processed.add(other)
            for syn, sim in self.find_synonymous_entities(ekey):
                for other in self.entity_to_propositions_map.get(syn, []):
                    if other != prop_key and other not in processed and (
                        prop_set is None or other in prop_set
                    ):
                        connected.append((other, [{"type": "synonym", "entity1": ekey,
                                                   "entity2": syn, "similarity": sim}]))
                    processed.add(other)
        self.connected_propositions_cache[prop_key] = (connected, processed)
        return connected, processed

    # --------------------------------------------------------------- scoring
    def batch_score_paths(self, paths: List[Dict], query_embedding: np.ndarray) -> List[float]:
        if not paths:
            return []
        prop_lists = [p["propositions"] for p in paths]
        if self.embedding_combination == "concatenate":
            texts = [" ".join(self.get_proposition_text(k) for k in props) for props in prop_lists]
            combined = self.rag.embedding_model.batch_encode(texts, norm=True)
        else:
            batch = np.array([list(self.get_proposition_embeddings(props)) for props in prop_lists])
            if self.embedding_combination == "average":
                combined = batch.mean(axis=1)
            elif self.embedding_combination == "weighted_average":
                w = np.linspace(0.5, 1.0, batch.shape[1])
                w = w / w.sum()
                combined = (batch * w.reshape(1, -1, 1)).sum(axis=1)
            elif self.embedding_combination == "max_pool":
                idx = np.argmax(np.abs(batch), axis=1)
                combined = np.take_along_axis(batch, idx[:, None, :], axis=1).squeeze(1)
            elif self.embedding_combination == "attention":
                qv = query_embedding.squeeze()
                att = np.exp(batch @ qv)
                att = att / att.sum(axis=1, keepdims=True)
                combined = (batch * att[:, :, None]).sum(axis=1)
            else:
                combined = batch.mean(axis=1)
            combined = combined / np.clip(np.linalg.norm(combined, axis=1, keepdims=True), 1e-12, None)
        return (combined @ query_embedding.squeeze()).tolist()

    # --------------------------------------------------------------- search
    def find_paths(self, query: str, prop_set: List[str] = None):
        q = self.rag.embedding_model.batch_encode(
            query, instruction=self.rag.config.embedding_query_instruction, norm=True,
            input_type="query",
        )
        if prop_set is None:
            all_keys = self.rag.proposition_embedding_store.get_all_ids()
            scores = (self.rag.all_proposition_embeddings @ q.T).squeeze()
        else:
            all_keys = prop_set
            scores = (self.get_proposition_embeddings(all_keys) @ q.T).squeeze()
        scores = np.atleast_1d(scores)

        top_k = min(self.rag.config.initial_proposition_seeds, len(scores))
        top_idx = np.argpartition(scores, -top_k)[-top_k:]
        top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
        beam = [{"propositions": [all_keys[i]], "connections": [], "score": float(scores[i])}
                for i in top_idx]

        initial_props = [(p["propositions"][0], []) for p in beam[:3]]
        beam = beam[:self.beam_width]
        all_paths, seen = [], set()
        for p in beam:
            fs = frozenset(p["propositions"])
            if fs not in seen:
                seen.add(fs)
                all_paths.append(p.copy())

        for depth in range(2, self.max_path_length + 1):
            new_candidates = []
            by_last = defaultdict(list)
            for p in beam:
                by_last[p["propositions"][-1]].append(p)
            for last_prop, paths_here in by_last.items():
                connected, conn_set = self.find_connected_propositions(last_prop, prop_set)
                connected = list(connected)
                for ip in initial_props:
                    if last_prop != ip[0] and ip[0] not in conn_set:
                        connected.append(ip)
                for path in paths_here:
                    for next_prop, conns in connected:
                        if next_prop in path["propositions"]:
                            continue
                        new_path = {
                            "propositions": path["propositions"] + [next_prop],
                            "connections": path["connections"] + [
                                {"from_prop": last_prop, "to_prop": next_prop,
                                 "entity_connections": conns}],
                        }
                        fs = frozenset(new_path["propositions"])
                        if fs in seen:
                            continue
                        seen.add(fs)
                        new_path["score"] = 0.0
                        new_candidates.append(new_path)
            if not new_candidates:
                break
            for i, s in enumerate(self.batch_score_paths(new_candidates, q)):
                new_candidates[i]["score"] = float(s)
            new_candidates.sort(key=lambda x: x["score"], reverse=True)

            if self.second_stage_filter_k > 0 and len(new_candidates) > self.beam_width:
                first = new_candidates[: self.second_stage_filter_k]
                # The second-stage re-rank refines with "concatenate" (re-embeds joined
                # path text). That means one fresh embedding API call per candidate;
                # under an online rate-limited embedder it dominates latency/cost, so
                # only do it for a local embedder. Online keeps the pooled scores.
                if not self.rag.config.embedding_is_online:
                    saved = self.embedding_combination
                    self.embedding_combination = "concatenate"
                    for i, s in enumerate(self.batch_score_paths(first, q)):
                        first[i]["score"] = float(s)
                    self.embedding_combination = saved
                first.sort(key=lambda x: x["score"], reverse=True)
                new_candidates = first

            beam = new_candidates[: self.beam_width]
            all_paths.extend(p.copy() for p in beam)

        all_paths.sort(key=lambda x: x["score"], reverse=True)
        return self._post_process(all_paths)

    def _post_process(self, all_paths):
        out = []
        for path in all_paths:
            out.append({
                "proposition_keys": path["propositions"],
                "proposition_texts": [self.get_proposition_text(k) for k in path["propositions"]],
                "connections": [{
                    "entity_connections": [{
                        "entity1": self.get_entity_text(c["entity1"]),
                        "entity2": self.get_entity_text(c["entity2"]),
                        "type": c["type"], "similarity": c.get("similarity", 1.0),
                    } for c in conn["entity_connections"]],
                } for conn in path["connections"]],
                "score": path.get("score", 0.0),
            })
        return out

    # --------------------------------------------------------- path → entities
    def get_entities_from_paths(self, paths) -> List[Tuple[str, List[float]]]:
        entity_scores = defaultdict(list)
        for path in paths:
            ps = path["score"]
            for conn in path.get("connections", []):
                for c in conn["entity_connections"]:
                    if c["entity1"] != c["entity2"]:
                        entity_scores[compute_mdhash_id(c["entity2"], prefix="entity-")].append(ps)
            for prop_key in path["proposition_keys"]:
                for ekey in self.find_entities_in_proposition(prop_key):
                    entity_scores[ekey].append(ps)
        return sorted(entity_scores.items(), key=lambda x: np.sum(x[1]), reverse=True)


In [ ]:
%%writefile /content/proprag_poc/embedding/__init__.py


In [ ]:
%%writefile /content/proprag_poc/embedding/encoder.py
"""Pluggable embedding encoder with an on-disk vector cache and usage
instrumentation.

Colab build: the ``sentence_transformers`` backend loads a local HuggingFace
model (default ``nvidia/NV-Embed-v2``). NV-Embed-v2 is a 7.85B-parameter model
that does NOT fit a free-tier T4 (15GB) in fp16, so it is loaded in 8-bit via
``bitsandbytes`` (``PROPRAG_EMBED_8BIT=1``, the default). A CPU fp32 fallback is
used if the 8-bit load fails. ``unload()`` frees the model from VRAM so the GGUF
chat model can take the whole GPU in the next phase.

The cache + ``batch_encode`` logic is byte-for-byte identical to the desktop
version so cache keys (model|input_type|instruction|text) stay stable across
phases: Phase B fills the cache while embedding; nothing re-encodes later.
"""

from __future__ import annotations

import gc
import hashlib
import logging
import os
import sqlite3
import threading
import time
from typing import List, Optional

import numpy as np

from ..config import POCConfig
from ..core.metrics import CallRecord, get_usage_tracker
from ..llm.rate_limiter import get_rate_limiter

logger = logging.getLogger(__name__)


def _env_flag(name: str, default: bool) -> bool:
    v = os.environ.get(name)
    if v is None:
        return default
    return v.strip().lower() not in ("0", "false", "no", "")


class EmbeddingModel:
    def __init__(self, config: POCConfig):
        self.config = config
        self._backend = config.embedding_backend
        self._dim: Optional[int] = None
        self._model = None
        self._client = None
        self._append_eos = False  # NV-Embed-v2 (sentence-transformers path) needs a trailing EOS
        self._nv_native = False   # NV-Embed-v2 loaded via transformers (its own .encode)
        self._max_len = 512
        self._nv_batch = 4
        self._tracker = get_usage_tracker()
        self._limiter = (
            get_rate_limiter(config.rpm_limit) if config.embedding_is_online else None
        )
        self._cache_path = os.path.join(config.data_dir, "embedding_cache.sqlite")
        self._lock = threading.Lock()
        self._init_cache()
        self._init_backend()

    # ----------------------------------------------------------------- setup
    def _init_backend(self):
        if self._backend == "sentence_transformers":
            self._load_sentence_transformer()
        elif self._backend in ("nvidia", "ollama", "openai"):
            from openai import OpenAI

            base = self.config.embedding_base_url or (
                "http://localhost:11434/v1" if self._backend == "ollama" else None
            )
            self._client = OpenAI(
                base_url=base,
                api_key=self.config.embedding_api_key or "not-needed",
                timeout=self.config.request_timeout,
            )
        else:
            raise ValueError(f"Unknown embedding backend: {self._backend}")

    def _load_sentence_transformer(self):
        import torch

        name = self.config.embedding_model
        is_nv_embed = "nv-embed" in name.lower()
        want_8bit = _env_flag("PROPRAG_EMBED_8BIT", True) and torch.cuda.is_available()
        self._nv_native = False          # NV-Embed-v2 loaded via transformers (its own .encode)
        self._max_len = int(os.environ.get("PROPRAG_EMBED_MAX_LEN", "512"))
        self._nv_batch = int(os.environ.get("PROPRAG_EMBED_BATCH", "4"))

        # Preferred path for NV-Embed-v2 on a T4: load in 8-bit straight through
        # transformers (device_map handles placement; no SentenceTransformer .to()
        # which rejects 8-bit models) and use the model's native encode().
        if is_nv_embed and want_8bit:
            try:
                from transformers import AutoModel, BitsAndBytesConfig

                bnb = BitsAndBytesConfig(load_in_8bit=True)
                logger.info("Loading NV-Embed-v2 in 8-bit (bitsandbytes) via transformers")
                model = AutoModel.from_pretrained(
                    name,
                    trust_remote_code=True,
                    quantization_config=bnb,
                    device_map="auto",
                    torch_dtype=torch.float16,
                )
                model.eval()
                self._model = model
                self._nv_native = True
                self._dim = int(getattr(model.config, "hidden_size", 4096))
                return
            except Exception as e:  # noqa: BLE001 - fall through to sentence-transformers/CPU
                logger.warning(
                    "8-bit NV-Embed load failed (%s); falling back to CPU sentence-transformers.", e
                )

        from sentence_transformers import SentenceTransformer

        # If 8-bit was wanted but unavailable, use CPU so the (large) model still
        # fits without competing with the GGUF model for T4 VRAM.
        device = "cpu" if want_8bit else ("cuda" if torch.cuda.is_available() else "cpu")
        logger.info("Loading embedder %s via sentence-transformers on %s", name, device)
        model = SentenceTransformer(name, trust_remote_code=True, device=device)
        if is_nv_embed:
            model.max_seq_length = self._max_len
            try:
                model.tokenizer.padding_side = "right"
            except Exception:  # noqa: BLE001
                pass
            self._append_eos = True
        self._model = model
        self._dim = model.get_sentence_embedding_dimension()

    def unload(self):
        """Free the local model from (V)RAM so the next phase's model can load."""
        import torch

        self._model = None
        self._client = None
        self._dim = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        logger.info("Embedding model unloaded; VRAM freed.")

    def _init_cache(self):
        with sqlite3.connect(self._cache_path) as conn:
            conn.execute(
                "CREATE TABLE IF NOT EXISTS vec (key TEXT PRIMARY KEY, dim INT, data BLOB)"
            )

    @property
    def embedding_dim(self) -> int:
        if self._dim is None:
            self._dim = self.batch_encode(["dimension probe"]).shape[1]
        return self._dim

    # ----------------------------------------------------------------- cache
    def _key(self, text: str, instruction: str, input_type: str) -> str:
        # input_type changes nv-embedqa output, so it is part of the cache key.
        raw = f"{self.config.embedding_model}|{input_type}|{instruction}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _cache_get_many(self, keys):
        out = {}
        with self._lock, sqlite3.connect(self._cache_path) as conn:
            conn.row_factory = None
            qmarks = ",".join("?" * len(keys))
            for k, dim, blob in conn.execute(
                f"SELECT key, dim, data FROM vec WHERE key IN ({qmarks})", keys
            ):
                out[k] = np.frombuffer(blob, dtype=np.float32).reshape(dim)
        return out

    def _cache_put_many(self, items):
        with self._lock, sqlite3.connect(self._cache_path) as conn:
            conn.executemany(
                "INSERT OR REPLACE INTO vec (key, dim, data) VALUES (?, ?, ?)",
                [(k, v.shape[0], v.astype(np.float32).tobytes()) for k, v in items],
            )

    # ---------------------------------------------------------------- encode
    def batch_encode(
        self,
        texts,
        instruction: str = "",
        norm: Optional[bool] = None,
        use_cache: bool = True,
        input_type: str = "passage",
    ) -> np.ndarray:
        """Encode texts. ``input_type`` is "passage" for stored docs, "query" for
        search queries (only meaningful for nv-embedqa models)."""
        if isinstance(texts, str):
            texts = [texts]
        if len(texts) == 0:
            return np.zeros((0, self.embedding_dim), dtype=np.float32)
        norm = self.config.embedding_normalize if norm is None else norm

        keys = [self._key(t, instruction, input_type) for t in texts]
        cached = self._cache_get_many(keys) if use_cache else {}

        if use_cache and cached:
            self._tracker.record(
                CallRecord(
                    kind="embed",
                    model=self.config.embedding_model,
                    cache_hit=True,
                    batch_size=len(cached),
                )
            )

        todo_idx = [i for i, k in enumerate(keys) if k not in cached]
        if todo_idx:
            todo_texts = [instruction + texts[i] for i in todo_idx]
            fresh = self._encode_raw(todo_texts, input_type)
            new_items = []
            for j, i in enumerate(todo_idx):
                cached[keys[i]] = fresh[j]
                new_items.append((keys[i], fresh[j]))
            if use_cache:
                self._cache_put_many(new_items)

        vecs = np.stack([cached[k] for k in keys]).astype(np.float32)
        if norm:
            norms = np.linalg.norm(vecs, axis=1, keepdims=True)
            vecs = vecs / np.clip(norms, 1e-12, None)
        return vecs

    def _encode_raw(self, texts: List[str], input_type: str) -> np.ndarray:
        if self._backend == "sentence_transformers":
            if self._model is None:
                raise RuntimeError(
                    "Embedding model is unloaded but a cache miss occurred. In the "
                    "3-phase Colab flow every text must be encoded during the "
                    "embedding phase; a miss here means a text was not pre-encoded."
                )
            if self._nv_native:
                import torch

                out = []
                bs = max(1, self._nv_batch)
                for start in range(0, len(texts), bs):
                    with torch.no_grad():
                        # NV-Embed-v2's own encode() handles EOS + pooling; the
                        # per-text instruction is already prepended by batch_encode,
                        # so pass instruction="" here.
                        emb = self._model.encode(
                            texts[start : start + bs], instruction="", max_length=self._max_len
                        )
                    out.append(np.asarray(emb.detach().to(torch.float32).cpu().numpy(), dtype=np.float32))
                vecs = np.vstack(out) if out else np.zeros((0, self.embedding_dim), dtype=np.float32)
            else:
                enc_texts = texts
                if self._append_eos:
                    eos = self._model.tokenizer.eos_token or ""
                    enc_texts = [t + eos for t in texts]
                vecs = np.asarray(
                    self._model.encode(
                        enc_texts,
                        batch_size=self.config.embedding_batch_size,
                        normalize_embeddings=False,
                        show_progress_bar=False,
                    ),
                    dtype=np.float32,
                )
            self._tracker.record(
                CallRecord(
                    kind="embed", model=self.config.embedding_model, batch_size=len(texts)
                )
            )
            return vecs

        # OpenAI-compatible / NVIDIA: send in capped batches, one rate-limit slot each.
        out: List[np.ndarray] = []
        bs = max(1, self.config.embedding_batch_size)
        extra = {}
        if self.config.embedding_needs_input_type:
            extra = {"input_type": input_type, "truncate": "END"}
        for start in range(0, len(texts), bs):
            chunk = texts[start : start + bs]
            rate_wait = self._limiter.acquire() if self._limiter is not None else 0.0
            t0 = time.monotonic()
            resp = self._client.embeddings.create(
                model=self.config.embedding_model,
                input=chunk,
                **({"extra_body": extra} if extra else {}),
            )
            latency = time.monotonic() - t0
            usage = getattr(resp, "usage", None)
            tokens = int(getattr(usage, "total_tokens", 0) or getattr(usage, "prompt_tokens", 0) or 0)
            self._tracker.record(
                CallRecord(
                    kind="embed",
                    model=self.config.embedding_model,
                    prompt_tokens=tokens,
                    latency_s=latency,
                    rate_wait_s=rate_wait,
                    batch_size=len(chunk),
                )
            )
            out.extend(np.asarray(d.embedding, dtype=np.float32) for d in resp.data)
        return np.vstack(out)


In [ ]:
%%writefile /content/proprag_poc/llm/__init__.py


In [ ]:
%%writefile /content/proprag_poc/llm/client.py
"""Chat LLM client with a SQLite response cache, retry, parallel inference,
a shared request rate limiter, and token-usage instrumentation.

Two backends are dispatched transparently:
  * ``google``  -> the native google-genai SDK (Gemini): system prompt via
    ``system_instruction``, roles ``user`` / ``model``.
  * everything else (``nvidia``, ``openrouter``, ``koboldcpp``, ``ollama``,
    ``vllm``) -> the OpenAI-compatible Chat Completions API via the ``openai`` SDK.

Online backends (NVIDIA / Gemini / OpenRouter) acquire a slot from the shared
``RateLimiter`` before each network call and report token usage + latency to the
``UsageTracker`` so the Compare view can attribute cost per RAG system.
"""

from __future__ import annotations

import hashlib
import json
import logging
import os
import sqlite3
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from typing import Callable, Dict, List, Optional, Tuple

from ..config import POCConfig
from ..core.metrics import CallRecord, get_usage_tracker
from .rate_limiter import get_rate_limiter

logger = logging.getLogger(__name__)


def _split_system(messages: List[Dict[str, str]]) -> Tuple[Optional[str], List[Dict[str, str]]]:
    """Separate system messages from the turn list and remap roles for Gemini."""
    system_parts, turns = [], []
    for m in messages:
        role = m.get("role")
        if role == "system":
            system_parts.append(m["content"])
        else:
            gem_role = "model" if role == "assistant" else "user"
            turns.append({"role": gem_role, "content": m["content"]})
    system = "\n\n".join(system_parts) if system_parts else None
    return system, turns


class LLMClient:
    """Cached chat client (Gemini or OpenAI-compatible) with rate limiting + metrics."""

    def __init__(self, config: POCConfig):
        self.config = config
        self._backend = config.llm_backend
        self._tracker = get_usage_tracker()
        self._limiter = get_rate_limiter(config.rpm_limit) if config.llm_is_online else None
        # Some OpenAI-compatible servers reject response_format; drop it after a failure.
        self._json_response_format_ok = True

        if self._backend == "google":
            from google import genai  # lazy: keep package optional at import time

            self._genai = genai
            from google.genai import types

            self._types = types
            self.client = genai.Client(api_key=config.llm_api_key)
        else:
            from openai import OpenAI

            self.client = OpenAI(
                base_url=config.llm_base_url,
                api_key=config.llm_api_key or "not-needed",
                timeout=config.request_timeout,
            )

        self._cache_path = os.path.join(config.data_dir, "llm_cache.sqlite")
        self._lock = threading.Lock()
        self._call_n = 0  # completed network calls, for progress logging
        self._init_cache()

    # ----------------------------------------------------------------- cache
    def _init_cache(self):
        with sqlite3.connect(self._cache_path) as conn:
            conn.execute(
                "CREATE TABLE IF NOT EXISTS cache (key TEXT PRIMARY KEY, value TEXT)"
            )

    def _cache_key(self, system, turns, temperature, max_tokens, json_mode) -> str:
        payload = json.dumps(
            {
                "system": system,
                "messages": turns,
                "backend": self._backend,
                "model": self.config.llm_model,
                "temperature": temperature,
                "max_tokens": max_tokens,
                "json": json_mode,
            },
            sort_keys=True,
        )
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    def _cache_get(self, key: str) -> Optional[str]:
        with self._lock, sqlite3.connect(self._cache_path) as conn:
            row = conn.execute("SELECT value FROM cache WHERE key=?", (key,)).fetchone()
        return row[0] if row else None

    def _cache_put(self, key: str, value: str):
        with self._lock, sqlite3.connect(self._cache_path) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO cache (key, value) VALUES (?, ?)", (key, value)
            )

    # ----------------------------------------------------------------- infer
    def infer(
        self,
        messages: List[Dict[str, str]],
        temperature: Optional[float] = None,
        json_mode: bool = False,
        use_cache: bool = True,
        max_completion_tokens: Optional[int] = None,
        response_checker: Optional[Callable[[str], bool]] = None,
    ) -> Tuple[str, Dict, bool]:
        """Return ``(content, metadata, cache_hit)``."""
        temperature = self.config.temperature if temperature is None else temperature
        max_tokens = max_completion_tokens or self.config.max_completion_tokens
        system, turns = _split_system(messages)
        key = self._cache_key(system, turns, temperature, max_tokens, json_mode)

        if use_cache:
            cached = self._cache_get(key)
            if cached is not None:
                self._tracker.record(
                    CallRecord(kind="chat", model=self.config.llm_model, cache_hit=True)
                )
                return cached, {"finish_reason": "cached"}, True

        last_err: Optional[Exception] = None
        attempt_temp = temperature
        rate_wait_total = 0.0
        for attempt in range(self.config.max_retry_attempts):
            if self._limiter is not None:
                rate_wait_total += self._limiter.acquire()
            with self._lock:
                call_no = self._call_n + 1
            logger.info(
                "chat #%d: request sent (attempt %d/%d)...",
                call_no, attempt + 1, self.config.max_retry_attempts,
            )
            t0 = time.monotonic()
            try:
                content, prompt_tok, compl_tok = self._call_backend(
                    system, turns, attempt_temp, max_tokens, json_mode
                )
                latency = time.monotonic() - t0
                if not content:
                    raise RuntimeError("empty response (possibly blocked or truncated)")
                if response_checker is not None and not response_checker(content):
                    attempt_temp = min(1.0, attempt_temp + 0.1)
                    last_err = ValueError("response_checker rejected output")
                    continue
                self._tracker.record(
                    CallRecord(
                        kind="chat",
                        model=self.config.llm_model,
                        prompt_tokens=prompt_tok,
                        completion_tokens=compl_tok,
                        latency_s=latency,
                        rate_wait_s=rate_wait_total,
                    )
                )
                with self._lock:
                    self._call_n += 1
                    n = self._call_n
                logger.info(
                    "chat #%d done in %.1fs (%d+%d tok)%s",
                    n, latency, prompt_tok, compl_tok,
                    f", waited {rate_wait_total:.1f}s" if rate_wait_total > 0.05 else "",
                )
                if use_cache:
                    self._cache_put(key, content)
                return content, {"finish_reason": "stop"}, False
            except Exception as e:  # noqa: BLE001 - retried below
                last_err = e
                logger.warning("LLM call failed (attempt %d): %s", attempt + 1, e)
        raise RuntimeError(f"LLM inference failed after retries: {last_err}")

    # ------------------------------------------------------ backend dispatch
    def _call_backend(self, system, turns, temperature, max_tokens, json_mode):
        if self._backend == "google":
            return self._call_google(system, turns, temperature, max_tokens, json_mode)
        return self._call_openai(system, turns, temperature, max_tokens, json_mode)

    def _call_google(self, system, turns, temperature, max_tokens, json_mode):
        types = self._types
        contents = [
            types.Content(role=t["role"], parts=[types.Part.from_text(text=t["content"])])
            for t in turns
        ]
        cfg = types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
            system_instruction=system,
            response_mime_type="application/json" if json_mode else None,
        )
        resp = self.client.models.generate_content(
            model=self.config.llm_model, contents=contents, config=cfg
        )
        content = (resp.text or "").strip()
        usage = getattr(resp, "usage_metadata", None)
        prompt_tok = int(getattr(usage, "prompt_token_count", 0) or 0)
        compl_tok = int(getattr(usage, "candidates_token_count", 0) or 0)
        return content, prompt_tok, compl_tok

    def _call_openai(self, system, turns, temperature, max_tokens, json_mode):
        # OpenAI-compatible expects roles user/assistant (turns use user/model).
        msgs = []
        if system:
            msgs.append({"role": "system", "content": system})
        for t in turns:
            role = "assistant" if t["role"] == "model" else "user"
            msgs.append({"role": role, "content": t["content"]})

        kwargs = dict(
            model=self.config.llm_model,
            messages=msgs,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        if self.config.seed is not None:
            kwargs["seed"] = self.config.seed
        use_rf = json_mode and self._json_response_format_ok
        if use_rf:
            kwargs["response_format"] = {"type": "json_object"}
        try:
            resp = self.client.chat.completions.create(**kwargs)
        except Exception as e:  # noqa: BLE001
            # Some NIM endpoints reject response_format; retry once without it and
            # disable it for the rest of the session (prompt + lenient parsing cope).
            if use_rf and "response_format" in str(e).lower():
                self._json_response_format_ok = False
                kwargs.pop("response_format", None)
                resp = self.client.chat.completions.create(**kwargs)
            else:
                raise
        content = (resp.choices[0].message.content or "").strip()
        usage = getattr(resp, "usage", None)
        prompt_tok = int(getattr(usage, "prompt_tokens", 0) or 0)
        compl_tok = int(getattr(usage, "completion_tokens", 0) or 0)
        return content, prompt_tok, compl_tok

    # --------------------------------------------------------------- parallel
    def infer_many(
        self,
        message_list: List[List[Dict[str, str]]],
        json_mode: bool = False,
        max_completion_tokens: Optional[int] = None,
        desc: str = "LLM",
    ) -> List[str]:
        """Run many chat calls in parallel; preserves input order."""
        results: List[Optional[str]] = [None] * len(message_list)

        def _work(i: int):
            content, _, _ = self.infer(
                message_list[i],
                json_mode=json_mode,
                max_completion_tokens=max_completion_tokens,
            )
            results[i] = content

        with ThreadPoolExecutor(max_workers=self.config.llm_max_workers) as ex:
            list(ex.map(_work, range(len(message_list))))
        return [r if r is not None else "" for r in results]


In [ ]:
%%writefile /content/proprag_poc/llm/rate_limiter.py
"""Process-wide request rate limiter (sliding 60s window).

A single shared limiter caps the request rate against the online provider
(NVIDIA) so it is not hit faster than its free-tier allowance. Only *real
network calls* should acquire a slot — cache hits and local backends must skip
it (pass through ``acquire`` only when needed).

By default this only guards the chat LLM client: embeddings run locally
(``sentence_transformers``) and never acquire a slot. If the embedding backend
is switched to an online one (``nvidia``/``openai``), it shares this same
limiter with chat, keyed by ``config.rpm_limit``, and batches count as a single
slot regardless of how many texts they carry.
"""

from __future__ import annotations

import logging
import threading
import time
from collections import deque
from typing import Dict, Optional

logger = logging.getLogger(__name__)

_WINDOW_SECONDS = 60.0


class RateLimiter:
    """Thread-safe sliding-window limiter: at most ``rpm`` acquisitions per 60s."""

    def __init__(self, rpm: int):
        self.rpm = max(1, int(rpm))
        self._times: deque[float] = deque()
        self._lock = threading.Lock()

    def acquire(self) -> float:
        """Block until a slot is free in the trailing 60s window.

        Returns the seconds spent waiting (0.0 if no wait was needed).
        """
        waited = 0.0
        while True:
            with self._lock:
                now = time.monotonic()
                # Drop timestamps older than the window.
                while self._times and now - self._times[0] >= _WINDOW_SECONDS:
                    self._times.popleft()
                if len(self._times) < self.rpm:
                    self._times.append(now)
                    return waited
                # Otherwise wait until the oldest timestamp exits the window.
                sleep_for = _WINDOW_SECONDS - (now - self._times[0]) + 0.01
            logger.info(
                "rate limit reached (%d/%d in 60s) - waiting %.1fs",
                self.rpm, self.rpm, max(0.0, sleep_for),
            )
            time.sleep(max(0.0, sleep_for))
            waited += max(0.0, sleep_for)


# ----------------------------------------------------------------- singleton
_INSTANCES: Dict[int, RateLimiter] = {}
_INSTANCES_LOCK = threading.Lock()


def get_rate_limiter(rpm: int) -> RateLimiter:
    """Return the shared limiter for a given RPM (one instance per rpm value).

    Chat and embedding clients call this with the same ``config.rpm_limit`` so they
    share one window.
    """
    with _INSTANCES_LOCK:
        inst = _INSTANCES.get(rpm)
        if inst is None:
            inst = RateLimiter(rpm)
            _INSTANCES[rpm] = inst
        return inst


In [ ]:
%%writefile /content/proprag_poc/llm/prompts.py
"""Prompt templates: NER, proposition extraction, JSON repair, RAG QA, query
contextualization. NER + proposition prompts are ported from the reference
PropRAG templates; the QA and contextualization prompts are POC additions.
"""

from __future__ import annotations

from typing import Dict, List

# --------------------------------------------------------------------------- NER
_NER_SYSTEM = """You are an expert at named-entity recognition. Extract every named
entity (people, organizations, locations, dates, products, events, works,
quantities and other proper noun phrases) from the passage. Be comprehensive.
Respond ONLY with a JSON object of the form {"entities": ["...", "..."]}."""

_NER_USER = """Passage:
```
{passage}
```
Extract the named entities as JSON."""


def ner_messages(passage: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": _NER_SYSTEM},
        {"role": "user", "content": _NER_USER.format(passage=passage)},
    ]


# ----------------------------------------------------------------- propositions
_PROPOSITION_SYSTEM = """Your task is to break a passage into precise, atomic propositions
using ONLY a provided list of named entities. A proposition is a fully
contextualized statement expressing a single unit of meaning.

For each proposition:
1. Extract a complete, standalone statement that preserves full context.
2. Use ONLY entities from the named_entities list - do not introduce new entities.
3. Ensure each proposition contains only ONE claim or relationship.
4. Be specific about which entities are involved in each relationship.
5. Preserve temporal and causal relationships.

Respond ONLY with a JSON object:
{"propositions": [{"text": "...", "entities": ["...", "..."]}, ...]}
where each "entities" array contains only entities from the named_entities list."""

_PROPOSITION_USER = """Passage:
```
{passage}
```

Named entities: {named_entities}

Break the passage into atomic propositions as JSON."""


def proposition_messages(passage: str, named_entities_json: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": _PROPOSITION_SYSTEM},
        {
            "role": "user",
            "content": _PROPOSITION_USER.format(
                passage=passage, named_entities=named_entities_json
            ),
        },
    ]


# ------------------------------------------------------------------- JSON repair
_FIX_JSON_SYSTEM = """You repair malformed JSON. Return ONLY valid JSON that preserves
the intended structure and content. Do not add commentary."""


def fix_json_messages(broken: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": _FIX_JSON_SYSTEM},
        {"role": "user", "content": f"Fix this JSON:\n```\n{broken}\n```"},
    ]


# ------------------------------------------------------------------------ RAG QA
_QA_SYSTEM = """You answer questions using ONLY the provided context passages and the
prior conversation. Think briefly, then give a final answer. If the answer is not
in the context, say you don't know. End your reply with a line starting with
"Answer:" followed by the concise final answer."""


def qa_messages(
    question: str,
    passages: List[str],
    history: List[Dict[str, str]] | None = None,
) -> List[Dict[str, str]]:
    messages: List[Dict[str, str]] = [{"role": "system", "content": _QA_SYSTEM}]
    if history:
        for turn in history:
            role = "assistant" if turn.get("role") == "assistant" else "user"
            messages.append({"role": role, "content": turn["content"]})
    context = "\n\n".join(f"[Passage {i + 1}]\n{p}" for i, p in enumerate(passages))
    messages.append(
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}\nThought:",
        }
    )
    return messages


# -------------------------------------------------------- query contextualization
_CONTEXT_SYSTEM = """Given a conversation history and a follow-up question, rewrite the
follow-up into a standalone, self-contained search query that resolves all
pronouns and references using the history. Respond ONLY with a JSON object:
{"query": "<standalone query>"}. Do not answer the question."""


def contextualize_messages(
    question: str, history: List[Dict[str, str]]
) -> List[Dict[str, str]]:
    hist = "\n".join(f"{t['role']}: {t['content']}" for t in history)
    return [
        {"role": "system", "content": _CONTEXT_SYSTEM},
        {
            "role": "user",
            "content": f"Conversation:\n{hist}\n\nFollow-up question: {question}\n\n"
            "Rewrite as a standalone query (JSON).",
        },
    ]


In [ ]:
%%writefile /content/benchmark/__init__.py
"""PropRAG vs GraphRAG vs BaseRAG benchmark harness on 2WikiMultiHopQA."""


In [ ]:
%%writefile /content/benchmark/_bootstrap.py
"""Put the sibling ``proprag_poc`` package on sys.path.

In the Colab build both packages are written next to each other (e.g. under
``/content``): ``/content/benchmark`` and ``/content/proprag_poc``. This module
adds that shared parent directory to ``sys.path`` so ``import proprag_poc`` works
from anywhere. ``PROPRAG_MAIN`` still overrides the location if set.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

_env_main = os.environ.get("PROPRAG_MAIN")
if _env_main and (Path(_env_main) / "proprag_poc" / "config.py").is_file():
    _PROPRAG_PARENT = str(Path(_env_main).resolve())
else:
    # parents[1] of .../benchmark/_bootstrap.py is the directory that holds both
    # ``benchmark`` and ``proprag_poc``.
    _PROPRAG_PARENT = str(Path(__file__).resolve().parents[1])

if _PROPRAG_PARENT not in sys.path:
    sys.path.insert(0, _PROPRAG_PARENT)


In [ ]:
%%writefile /content/benchmark/bench_config.py
"""Benchmark configuration wrapping the reused POCConfig.

Desktop runs can still use Koboldcpp. Colab sets PROPRAG_LLM_BACKEND=llama_cpp
and PROPRAG_GGUF_MODEL_PATH so inference uses a downloaded GGUF model directly.
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Optional, Tuple

from . import _bootstrap  # noqa: F401 - side effect: proprag_poc on sys.path
from proprag_poc.config import POCConfig

# Backends POCConfig's preset table actually knows (see proprag_poc/config.py
# _LLM_PRESETS). "llama_cpp" is a benchmark-only value, not a POC preset.
_POC_BACKENDS = {"nvidia", "google", "koboldcpp", "ollama", "vllm", "openrouter"}

# Colab: the two 2wiki JSON files are downloaded into this directory. Overridable
# per-file via PROPRAG_DATASET_PATH / PROPRAG_CORPUS_PATH (read in __post_init__).
_DATASET_DIR = os.environ.get("PROPRAG_DATASET_DIR", "/content/datasets")
_DEFAULT_DATASET = os.path.join(_DATASET_DIR, "2wikimultihopqa.json")
_DEFAULT_CORPUS = os.path.join(_DATASET_DIR, "2wikimultihopqa_corpus.json")


@dataclass
class BenchmarkConfig:
    project_dir: str
    dataset_path: str = _DEFAULT_DATASET
    corpus_path: str = _DEFAULT_CORPUS

    n_questions: int = 50
    seed: int = 42
    pilot: Optional[int] = None

    recall_ks: Tuple[int, ...] = (2, 5, 10)
    retrieval_top_k: int = 10
    qa_top_k: int = 5
    qa_max_answer_tokens: int = 512

    extract_max_tokens: int = 1500
    summarize_max_tokens: int = 512
    report_max_tokens: int = 1000

    gr_entity_types: Tuple[str, ...] = (
        "person", "organization", "location", "event", "work", "date", "other",
    )
    gr_max_gleanings: int = 0
    gr_summarize_desc_threshold: int = 800
    gr_min_community_size: int = 2
    gr_report_max_input_chars: int = 12000
    gr_local_top_entities: int = 10
    gr_entity_sim_floor: float = 0.5
    gr_dense_blend: float = 0.5

    systems: Tuple[str, ...] = ("BaseRAG", "GraphRAG", "PropRAG")

    def __post_init__(self):
        self.project_dir = os.path.abspath(self.project_dir)
        self.dataset_path = os.environ.get("PROPRAG_DATASET_PATH", self.dataset_path)
        self.corpus_path = os.environ.get("PROPRAG_CORPUS_PATH", self.corpus_path)

    @property
    def data_dir(self) -> str:
        return os.path.join(self.project_dir, "data")

    def make_poc_config(self) -> POCConfig:
        # POCConfig's preset table only knows nvidia/google/koboldcpp/ollama/vllm/
        # openrouter; "llama_cpp" (Colab's direct in-process GGUF mode, no HTTP
        # server) isn't one of them and would KeyError in POCConfig.__post_init__.
        # Construct with a valid local preset so init succeeds, then relabel -
        # BenchLLMClient/check_backend both dispatch on poc_cfg.llm_backend ==
        # "llama_cpp" and never touch the preset's URL/model in that mode.
        requested = os.environ.get("PROPRAG_LLM_BACKEND", "koboldcpp")
        construct_backend = requested if requested in _POC_BACKENDS else "koboldcpp"
        poc_cfg = POCConfig(
            data_dir=self.data_dir,
            llm_backend=construct_backend,
            temperature=0.0,
            seed=self.seed,
            retrieval_top_k=self.retrieval_top_k,
            qa_top_k=self.qa_top_k,
            llm_max_workers=1,
            strip_reasoning=True,
        )
        if requested == "llama_cpp":
            poc_cfg.llm_backend = "llama_cpp"
        return poc_cfg


In [ ]:
%%writefile /content/benchmark/dataset.py
"""2WikiMultiHopQA loading, stratified subsetting, and corpus assembly.

Pure Python (no LLM, no embeddings) so it can be sanity-checked offline via
``python -m benchmark.dataset``.
"""

from __future__ import annotations

import hashlib
import json
import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

# Canonical type order: fixes largest-remainder tie-breaks and output sorting.
TYPE_ORDER = ("compositional", "comparison", "bridge_comparison", "inference")


@dataclass
class BenchQuestion:
    qid: str
    qtype: str
    question: str
    answer: str
    gold_titles: List[str]
    context_titles: List[str]

    @property
    def gold_answers(self) -> List[str]:
        """EM/F1 gold list. 2wiki gives one canonical answer per question."""
        return [self.answer]


# --------------------------------------------------------------------- loading
def load_questions(path: str) -> List[BenchQuestion]:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    out: List[BenchQuestion] = []
    for q in raw:
        gold = list(dict.fromkeys(sf[0] for sf in q.get("supporting_facts", [])))
        ctx = list(dict.fromkeys(c[0] for c in q.get("context", [])))
        out.append(
            BenchQuestion(
                qid=q["_id"],
                qtype=q["type"],
                question=q["question"],
                answer=q["answer"],
                gold_titles=gold,
                context_titles=ctx,
            )
        )
    return out


# ------------------------------------------------------------- allocation math
def _largest_remainder(counts: Dict[str, int], n: int) -> Dict[str, int]:
    """Proportionally allocate ``n`` slots across types; tie-break by TYPE_ORDER.

    Clamps each allocation to what the type actually has and redistributes any
    shortfall to types with spare capacity (largest remainder first).
    """
    types = [t for t in TYPE_ORDER if t in counts] + [
        t for t in counts if t not in TYPE_ORDER
    ]
    total = sum(counts.values())
    if total == 0 or n <= 0:
        return {t: 0 for t in types}

    raw = {t: counts[t] * n / total for t in types}
    alloc = {t: int(raw[t]) for t in types}
    order_index = {t: i for i, t in enumerate(types)}

    def frac_key(t: str) -> Tuple[float, int]:
        return (-(raw[t] - int(raw[t])), order_index[t])

    remainder = n - sum(alloc.values())
    for t in sorted(types, key=frac_key)[:remainder]:
        alloc[t] += 1

    # Clamp to availability, then redistribute any shortfall.
    for t in types:
        alloc[t] = min(alloc[t], counts[t])
    shortfall = n - sum(alloc.values())
    while shortfall > 0:
        spare = [t for t in types if alloc[t] < counts[t]]
        if not spare:
            break
        for t in sorted(spare, key=frac_key):
            if shortfall == 0:
                break
            alloc[t] += 1
            shortfall -= 1
    return alloc


def _stratified_pick(
    qs: List[BenchQuestion], n: int, seed: int
) -> List[BenchQuestion]:
    by_type: Dict[str, List[BenchQuestion]] = {}
    for q in qs:
        by_type.setdefault(q.qtype, []).append(q)
    counts = {t: len(v) for t, v in by_type.items()}
    alloc = _largest_remainder(counts, n)

    picked: List[BenchQuestion] = []
    rng = random.Random(seed)
    for t in sorted(by_type, key=lambda x: TYPE_ORDER.index(x) if x in TYPE_ORDER else 99):
        pool = sorted(by_type[t], key=lambda q: q.qid)  # deterministic pool order
        picked.extend(rng.sample(pool, alloc[t]))
    picked.sort(key=lambda q: (TYPE_ORDER.index(q.qtype) if q.qtype in TYPE_ORDER else 99, q.qid))
    return picked


def stratified_subset(qs: List[BenchQuestion], n: int, seed: int) -> List[BenchQuestion]:
    return _stratified_pick(qs, n, seed)


def pilot_subset(qs_subset: List[BenchQuestion], k: int, seed: int) -> List[BenchQuestion]:
    """Stratified ``k``-of-subset. Its questions are a subset of ``qs_subset`` so
    every pilot extraction becomes a cache hit during the full run."""
    return _stratified_pick(qs_subset, k, seed)


# ----------------------------------------------------------------- corpus build
def chunk_text(title: str, text: str) -> str:
    """One searchable chunk per document: the title on its own line, then body."""
    return f"{title}\n{text}"


def build_corpus(qs: List[BenchQuestion], corpus_path: str) -> Dict[str, str]:
    """title -> text for the union of the subset's context titles.

    Asserts every gold and context title exists in the corpus so Recall@k and QA
    context are never silently missing a gold passage.
    """
    with open(corpus_path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    full: Dict[str, str] = {}
    for doc in raw:
        title = doc["title"]
        if title not in full:  # dedupe by title, first wins
            full[title] = doc["text"]

    needed = set()
    for q in qs:
        needed.update(q.context_titles)
        needed.update(q.gold_titles)

    missing = sorted(t for t in needed if t not in full)
    if missing:
        raise ValueError(
            f"{len(missing)} needed titles absent from corpus, e.g. {missing[:3]}"
        )
    return {t: full[t] for t in sorted(needed)}


# -------------------------------------------------------------------- identity
def subset_hash(qs: List[BenchQuestion]) -> str:
    joined = "|".join(sorted(q.qid for q in qs))
    return hashlib.sha1(joined.encode("utf-8")).hexdigest()


def corpus_id(qs: List[BenchQuestion], tag: str) -> str:
    return f"2wiki_{tag}_{subset_hash(qs)[:10]}"


def type_counts(qs: List[BenchQuestion]) -> Dict[str, int]:
    counts: Dict[str, int] = {}
    for q in qs:
        counts[q.qtype] = counts.get(q.qtype, 0) + 1
    return counts


def write_manifest(
    run_dir: str, qs: List[BenchQuestion], titles: Dict[str, str], seed: int, cfg
) -> str:
    os.makedirs(run_dir, exist_ok=True)
    path = os.path.join(run_dir, "manifest.json")
    manifest = {
        "seed": seed,
        "n_questions": len(qs),
        "type_counts": type_counts(qs),
        "subset_hash": subset_hash(qs),
        "corpus_size": len(titles),
        "qids": [q.qid for q in qs],
        "dataset_path": cfg.dataset_path,
        "corpus_path": cfg.corpus_path,
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return path


# ---------------------------------------------------------------------- sanity
def _main() -> None:
    from .bench_config import BenchmarkConfig

    cfg = BenchmarkConfig(project_dir=os.path.dirname(os.path.dirname(__file__)))
    qs = load_questions(cfg.dataset_path)
    print(f"loaded {len(qs)} questions; full type counts: {type_counts(qs)}")

    subset = stratified_subset(qs, cfg.n_questions, cfg.seed)
    print(f"subset n={len(subset)} type counts: {type_counts(subset)}")

    pilot = pilot_subset(subset, 10, cfg.seed)
    print(f"pilot n={len(pilot)} type counts: {type_counts(pilot)}")
    assert set(p.qid for p in pilot).issubset(set(s.qid for s in subset)), "pilot not a subset"

    titles = build_corpus(subset, cfg.corpus_path)
    print(f"corpus subset size: {len(titles)} docs (expect ~380-420)")

    # Gold-title coverage.
    all_gold = set()
    for q in subset:
        all_gold.update(q.gold_titles)
    covered = sum(1 for t in all_gold if t in titles)
    print(f"gold titles: {len(all_gold)}, covered by corpus: {covered}")
    assert covered == len(all_gold), "some gold titles are missing from the corpus"
    print(f"corpus_id: {corpus_id(subset, 'n50')}")
    print("dataset sanity OK")


if __name__ == "__main__":
    _main()


In [ ]:
%%writefile /content/benchmark/evaluation.py
"""Answer and retrieval metrics.

``normalize_answer`` / EM / F1 are ported verbatim from PropRAG-main
(``utils/eval_utils.normalize_answer`` and ``evaluation/qa_eval``), stripped of
the ``BaseMetric`` scaffolding. Recall@k is the standard supporting-title recall.
Run ``python -m benchmark.evaluation`` to exercise the inline hand cases.
"""

from __future__ import annotations

import re
import string
from collections import Counter
from typing import Dict, List, Sequence


def normalize_answer(answer: str) -> str:
    """Lowercase, drop punctuation, drop a/an/the, collapse whitespace."""
    def remove_articles(text: str) -> str:
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text: str) -> str:
        return " ".join(text.split())

    def remove_punc(text: str) -> str:
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    return white_space_fix(remove_articles(remove_punc(answer.lower())))


def em_score(golds: Sequence[str], pred: str) -> float:
    """Exact match, max over the gold list (MRQA-style)."""
    npred = normalize_answer(pred)
    return max((1.0 if normalize_answer(g) == npred else 0.0) for g in golds) if golds else 0.0


def _f1(gold: str, pred: str) -> float:
    gold_tokens = normalize_answer(gold).split()
    pred_tokens = normalize_answer(pred).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def f1_score(golds: Sequence[str], pred: str) -> float:
    """Token-overlap F1, max over the gold list."""
    return max((_f1(g, pred) for g in golds), default=0.0)


def recall_at_k(
    gold_titles: Sequence[str],
    retrieved_titles: Sequence[str],
    ks: Sequence[int] = (2, 5, 10),
) -> Dict[str, float]:
    """For each k: |gold ∩ top-k retrieved| / |gold|."""
    gold = set(gold_titles)
    out: Dict[str, float] = {}
    for k in ks:
        if not gold:
            out[f"recall@{k}"] = 0.0
            continue
        topk = set(retrieved_titles[:k])
        out[f"recall@{k}"] = len(gold & topk) / len(gold)
    return out


def _main() -> None:
    assert normalize_answer("The  A.B.C.!") == "abc"
    assert em_score(["20 March 851"], "20 march 851.") == 1.0
    assert em_score(["Paris"], "London") == 0.0
    assert abs(f1_score(["barack obama"], "obama") - (2 * 1.0 * 0.5 / 1.5)) < 1e-9
    assert f1_score(["cat dog"], "cat dog") == 1.0
    r = recall_at_k(["A", "B"], ["A", "X", "B", "Y"], ks=(1, 2, 3))
    assert r == {"recall@1": 0.5, "recall@2": 0.5, "recall@3": 1.0}, r
    assert recall_at_k([], ["A"], ks=(2,)) == {"recall@2": 0.0}
    print("evaluation sanity OK")


if __name__ == "__main__":
    _main()


In [ ]:
%%writefile /content/benchmark/qa.py
"""Shared QA prompt + answer extraction for all three systems.

One identical prompt keeps the generation step fair; only the retrieved context
differs per system. Tuned for 2wiki short spans: brief reasoning, then a final
``Answer:`` line carrying the shortest span, which we parse back out for EM/F1.
"""

from __future__ import annotations

from typing import Dict, List, Tuple

_QA_SYSTEM = (
    "You answer questions using ONLY the provided context passages. Reason "
    "briefly, then finish with a line formatted exactly as 'Answer: <answer>' "
    "where <answer> is the shortest possible span (a name, date, or short "
    "phrase). If the answer is not in the context, write 'Answer: unknown'."
)


def qa_messages(question: str, context_texts: List[str]) -> List[Dict[str, str]]:
    context = "\n\n".join(f"[Passage {i + 1}]\n{p}" for i, p in enumerate(context_texts))
    return [
        {"role": "system", "content": _QA_SYSTEM},
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}\nReasoning:",
        },
    ]


def extract_answer(raw: str) -> str:
    """Pull the span after the last ``Answer:`` marker; first line only."""
    if not raw:
        return "unknown"
    tail = raw.rsplit("Answer:", 1)[1] if "Answer:" in raw else raw
    first_line = next((ln for ln in tail.strip().splitlines() if ln.strip()), "")
    return first_line.strip().strip('"').strip("'").strip(". ").strip() or "unknown"


def answer_question(llm, question: str, context_texts: List[str], cfg) -> Tuple[str, str]:
    """Return ``(short_answer, raw_response)``."""
    raw, _, _ = llm.infer(
        qa_messages(question, context_texts),
        json_mode=False,
        max_completion_tokens=cfg.qa_max_answer_tokens,
    )
    return extract_answer(raw), raw


In [ ]:
%%writefile /content/benchmark/results.py
"""Append-only, resume-safe results store (one JSON line per (question, system))."""

from __future__ import annotations

import json
import os
from typing import Dict, Set, Tuple


class ResultsStore:
    def __init__(self, run_dir: str):
        os.makedirs(run_dir, exist_ok=True)
        self.run_dir = run_dir
        self.path = os.path.join(run_dir, "results.jsonl")

    def done_keys(self) -> Set[Tuple[str, str]]:
        """(qid, system) pairs already recorded *successfully*.

        Error rows are excluded so a rerun retries transient failures; the report
        deduplicates by (qid, system) keeping the last row.
        """
        done: Set[Tuple[str, str]] = set()
        if not os.path.isfile(self.path):
            return done
        with open(self.path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if row.get("error"):
                    continue
                if "qid" in row and "system" in row:
                    done.add((row["qid"], row["system"]))
        return done

    def append(self, row: Dict) -> None:
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row) + "\n")
            f.flush()
            os.fsync(f.fileno())


In [ ]:
%%writefile /content/benchmark/report.py
"""Aggregate results.jsonl + index_usage.json into metrics.json + report.md.

Charts are optional: a guarded matplotlib import degrades to "no charts" if the
package is missing.
"""

from __future__ import annotations

import json
import logging
import os
from collections import defaultdict
from typing import Dict, List

logger = logging.getLogger(__name__)

_SYSTEMS = ("BaseRAG", "GraphRAG", "PropRAG")
_RECALL_KS = (2, 5, 10)


# ------------------------------------------------------------------- loading
def _load_rows(run_dir: str) -> List[Dict]:
    path = os.path.join(run_dir, "results.jsonl")
    latest: Dict = {}
    if not os.path.isfile(path):
        return []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get("error") or "qid" not in row or "system" not in row:
                continue
            latest[(row["qid"], row["system"])] = row  # last wins (dedupe)
    return list(latest.values())


def _load_json(path: str) -> Dict:
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def _mean(xs: List[float]) -> float:
    return sum(xs) / len(xs) if xs else 0.0


# --------------------------------------------------------------- aggregation
def _aggregate(rows: List[Dict]) -> Dict:
    by_system: Dict[str, List[Dict]] = defaultdict(list)
    for r in rows:
        by_system[r["system"]].append(r)

    metrics: Dict[str, Dict] = {}
    for system, srows in by_system.items():
        em = _mean([r.get("em", 0.0) for r in srows])
        f1 = _mean([r.get("f1", 0.0) for r in srows])
        recalls = {
            f"recall@{k}": _mean([r.get("recall", {}).get(f"recall@{k}", 0.0) for r in srows])
            for k in _RECALL_KS
        }
        usages = [r.get("usage", {}) for r in srows]
        metrics[system] = {
            "n": len(srows),
            "em": em,
            "f1": f1,
            **recalls,
            "mean_retrieval_latency_s": _mean([r.get("retrieval_latency_s", 0.0) for r in srows]),
            "mean_qa_latency_s": _mean([r.get("qa_latency_s", 0.0) for r in srows]),
            "mean_query_prompt_tokens": _mean([u.get("chat_prompt_tokens", 0) for u in usages]),
            "mean_query_completion_tokens": _mean([u.get("chat_completion_tokens", 0) for u in usages]),
            "mean_chat_calls_per_q": _mean([u.get("chat_calls", 0) for u in usages]),
            "by_type": _by_type(srows),
        }
    return metrics


def _by_type(srows: List[Dict]) -> Dict[str, Dict]:
    by_type: Dict[str, List[Dict]] = defaultdict(list)
    for r in srows:
        by_type[r.get("qtype", "?")].append(r)
    return {
        t: {
            "n": len(rs),
            "em": _mean([r.get("em", 0.0) for r in rs]),
            "f1": _mean([r.get("f1", 0.0) for r in rs]),
            "recall@5": _mean([r.get("recall", {}).get("recall@5", 0.0) for r in rs]),
        }
        for t, rs in sorted(by_type.items())
    }


# ------------------------------------------------------------------- markdown
def _fmt(x: float, pct: bool = False) -> str:
    return f"{x * 100:.1f}%" if pct else f"{x:.1f}"


def _render_md(metrics: Dict, index_usage: Dict, manifest: Dict, meta: Dict) -> str:
    systems = [s for s in _SYSTEMS if s in metrics] or list(metrics)
    lines: List[str] = ["# PropRAG vs GraphRAG vs BaseRAG — 2WikiMultiHopQA", ""]
    lines.append(f"Questions: {manifest.get('n_questions', '?')} "
                 f"(seed {manifest.get('seed', '?')}, subset {manifest.get('subset_hash', '?')[:10]}); "
                 f"corpus {manifest.get('corpus_size', '?')} docs.")
    lines.append("")

    # Answer + retrieval quality.
    lines.append("## Answer & retrieval quality")
    lines.append("")
    header = "| System | EM | F1 | R@2 | R@5 | R@10 |"
    lines += [header, "|---|---|---|---|---|---|"]
    for s in systems:
        m = metrics[s]
        lines.append(
            f"| {s} | {_fmt(m['em'], True)} | {_fmt(m['f1'], True)} | "
            f"{_fmt(m['recall@2'], True)} | {_fmt(m['recall@5'], True)} | {_fmt(m['recall@10'], True)} |"
        )
    lines.append("")

    # Per-question-type.
    lines.append("## By question type (EM / F1 / R@5)")
    lines.append("")
    lines += ["| Type | " + " | ".join(systems) + " |", "|---|" + "---|" * len(systems)]
    all_types = sorted({t for s in systems for t in metrics[s]["by_type"]})
    for t in all_types:
        cells = []
        for s in systems:
            bt = metrics[s]["by_type"].get(t)
            cells.append(
                f"{_fmt(bt['em'], True)} / {_fmt(bt['f1'], True)} / {_fmt(bt['recall@5'], True)}"
                if bt else "-"
            )
        lines.append(f"| {t} | " + " | ".join(cells) + " |")
    lines.append("")

    # Query cost.
    lines.append("## Query cost (per question)")
    lines.append("")
    lines += ["| System | chat calls | prompt tok | completion tok | retr latency (s) | QA latency (s) |",
              "|---|---|---|---|---|---|"]
    for s in systems:
        m = metrics[s]
        lines.append(
            f"| {s} | {m['mean_chat_calls_per_q']:.2f} | {m['mean_query_prompt_tokens']:.0f} | "
            f"{m['mean_query_completion_tokens']:.0f} | {m['mean_retrieval_latency_s']:.2f} | "
            f"{m['mean_qa_latency_s']:.2f} |"
        )
    lines.append("")

    # Index cost.
    lines.append("## Index cost (one-time)")
    lines.append("")
    lines += ["| System | chat calls | prompt tok | completion tok | embed texts | wall (s) | parse fails |",
              "|---|---|---|---|---|---|---|"]
    for s in systems:
        u = index_usage.get(s, {})
        lines.append(
            f"| {s} | {u.get('chat_calls', 0):.0f} | {u.get('chat_prompt_tokens', 0):.0f} | "
            f"{u.get('chat_completion_tokens', 0):.0f} | {u.get('embed_texts', 0):.0f} | "
            f"{u.get('wall_time_s', 0):.1f} | {u.get('parse_failures', 0)} |"
        )
    lines.append("")

    # Fairness footer.
    lines.append("## Fairness & configuration")
    lines.append("")
    lines.append(f"- Shared LLM: `{meta.get('llm_model', '?')}` @ `{meta.get('llm_backend', '?')}`; "
                 f"shared embedder: `{meta.get('embedding_model', '?')}`.")
    lines.append(f"- Shared chunk embeddings; qa_top_k={meta.get('qa_top_k', '?')}, "
                 f"retrieval_top_k={meta.get('retrieval_top_k', '?')}, seed={manifest.get('seed', '?')}.")
    lines.append(f"- GraphRAG gleanings={meta.get('gr_max_gleanings', '?')} "
                 "(Microsoft default is 1; 0 lowers index cost).")
    lines.append("- Recall@k uses retrieved documents; GraphRAG QA additionally uses community "
                 "reports + relationships via local-search context assembly.")
    total_wall = sum(index_usage.get(s, {}).get("wall_time_s", 0) for s in systems)
    lines.append(f"- Total index wall time: {total_wall:.1f}s.")
    lines.append("")
    return "\n".join(lines)


# --------------------------------------------------------------------- charts
def _charts(metrics: Dict, index_usage: Dict, run_dir: str) -> None:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except Exception:  # noqa: BLE001
        logger.info("matplotlib unavailable; skipping charts")
        return

    systems = [s for s in _SYSTEMS if s in metrics]
    if not systems:
        return
    charts_dir = os.path.join(run_dir, "charts")
    os.makedirs(charts_dir, exist_ok=True)

    # Quality bars.
    fig, ax = plt.subplots(figsize=(7, 4))
    import numpy as np
    x = np.arange(len(systems))
    width = 0.25
    for i, key in enumerate(("em", "f1", "recall@5")):
        ax.bar(x + (i - 1) * width, [metrics[s][key] for s in systems], width, label=key.upper())
    ax.set_xticks(x); ax.set_xticklabels(systems); ax.set_ylabel("score"); ax.legend()
    ax.set_title("Answer quality & Recall@5")
    fig.tight_layout(); fig.savefig(os.path.join(charts_dir, "quality.png"), dpi=120); plt.close(fig)

    # Index tokens.
    fig, ax = plt.subplots(figsize=(7, 4))
    prompt = [index_usage.get(s, {}).get("chat_prompt_tokens", 0) for s in systems]
    compl = [index_usage.get(s, {}).get("chat_completion_tokens", 0) for s in systems]
    ax.bar(x, prompt, width * 2, label="prompt")
    ax.bar(x, compl, width * 2, bottom=prompt, label="completion")
    ax.set_xticks(x); ax.set_xticklabels(systems); ax.set_ylabel("tokens"); ax.legend()
    ax.set_title("Index chat tokens per system")
    fig.tight_layout(); fig.savefig(os.path.join(charts_dir, "index_tokens.png"), dpi=120); plt.close(fig)
    logger.info("charts written to %s", charts_dir)


# ---------------------------------------------------------------------- build
def build(run_dir: str, make_charts: bool = True) -> Dict:
    rows = _load_rows(run_dir)
    index_usage = _load_json(os.path.join(run_dir, "index_usage.json"))
    manifest = _load_json(os.path.join(run_dir, "manifest.json"))
    meta = _load_json(os.path.join(run_dir, "run_meta.json"))

    metrics = _aggregate(rows)
    with open(os.path.join(run_dir, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    md = _render_md(metrics, index_usage, manifest, meta)
    with open(os.path.join(run_dir, "report.md"), "w", encoding="utf-8") as f:
        f.write(md)

    if make_charts:
        _charts(metrics, index_usage, run_dir)
    logger.info("report written to %s", os.path.join(run_dir, "report.md"))
    return metrics


In [ ]:
%%writefile /content/benchmark/usage.py
"""Per-phase token/latency attribution over the shared ``UsageTracker``.

The tracker's ``scope()`` is thread-local, and the reused ``Extractor.batch_extract``
runs in a thread pool, so index-time extraction lands in the default ``"index"``
scope regardless of any active scope. Index builds run sequentially, so each
phase's cost is the *delta* of the relevant scopes across the phase.

- BaseRAG / PropRAG index: their calls land in ``"index"`` -> delta of ``"index"``.
- GraphRAG index: worker calls set ``"index::GraphRAG"`` explicitly, main-thread
  calls land in ``"index"``; summing both scopes' deltas covers both paths.
"""

from __future__ import annotations

import time
from typing import Dict, List


def delta(before: Dict[str, float], after: Dict[str, float]) -> Dict[str, float]:
    """Field-wise ``after - before`` over ``Snapshot.as_dict()`` outputs."""
    keys = set(before) | set(after)
    return {k: after.get(k, 0) - before.get(k, 0) for k in keys}


def _add(into: Dict[str, float], other: Dict[str, float]) -> None:
    for k, v in other.items():
        into[k] = into.get(k, 0) + v


class IndexPhase:
    """Context manager capturing the usage delta of one index phase.

    ``self.usage`` is populated on exit: summed deltas of ``"index"`` plus any
    ``extra_scopes`` (e.g. ``"index::GraphRAG"``), and ``wall_time_s``.
    """

    def __init__(self, tracker, name: str, extra_scopes: List[str] = None):
        self._tracker = tracker
        self.name = name
        self._scopes = ["index"] + list(extra_scopes or [])
        self._before: Dict[str, Dict[str, float]] = {}
        self._t0 = 0.0
        self.usage: Dict[str, float] = {}

    def __enter__(self) -> "IndexPhase":
        self._before = {s: self._tracker.snapshot(s).as_dict() for s in self._scopes}
        self._t0 = time.monotonic()
        return self

    def __exit__(self, *exc) -> bool:
        wall = time.monotonic() - self._t0
        combined: Dict[str, float] = {}
        for s in self._scopes:
            _add(combined, delta(self._before[s], self._tracker.snapshot(s).as_dict()))
        combined["wall_time_s"] = round(wall, 3)
        self.usage = combined
        return False


In [ ]:
%%writefile /content/benchmark/llm_wrap.py
"""LLM client wrapper for gpt-oss inference.

Desktop runs can still use the original Koboldcpp/OpenAI-compatible path. Colab
uses PROPRAG_LLM_BACKEND=llama_cpp plus PROPRAG_GGUF_MODEL_PATH to load a
pre-quantized GGUF file directly with llama-cpp-python. That avoids a localhost
inference server and avoids quantizing while loading the model.
"""

from __future__ import annotations

import logging
import os
import threading
import time
import urllib.error
import urllib.request
from typing import Dict, List

from . import _bootstrap  # noqa: F401
from proprag_poc.config import POCConfig
from proprag_poc.core.metrics import CallRecord, get_usage_tracker
from proprag_poc.llm.client import LLMClient

_LLAMA_CPP_MODEL_LABEL = "gpt-oss-20b-gguf"

logger = logging.getLogger(__name__)

_FINAL_MARKER = "<|channel|>final<|message|>"
_TERMINATORS = ("<|end|>", "<|return|>", "<|start|>")


def strip_gpt_oss_reasoning(content: str) -> str:
    """Remove the gpt-oss Harmony reasoning channel, keep the final answer text."""
    if not content:
        return content
    if _FINAL_MARKER in content:
        content = content.split(_FINAL_MARKER)[-1]
        for term in _TERMINATORS:
            content = content.split(term)[0]
        return content.strip()
    stripped = content.lstrip()
    if stripped.startswith("analysis") and "assistantfinal" in stripped:
        return stripped.split("assistantfinal")[-1].strip()
    return content


def _env_int(name: str, default: int) -> int:
    try:
        return int(os.environ.get(name, default))
    except (TypeError, ValueError):
        return default


class BenchLLMClient(LLMClient):
    """LLMClient with gpt-oss post-processing and optional direct GGUF mode."""

    def __init__(self, config: POCConfig):
        self._direct_llama = config.llm_backend == "llama_cpp"
        if self._direct_llama:
            # Skip LLMClient.__init__ (it would build an OpenAI HTTP client for a
            # preset we're not using), but still wire up the SQLite response cache
            # and UsageTracker plumbing LLMClient.infer() normally provides -
            # otherwise every direct call would be uncached and untracked, which
            # would silently zero out this benchmark's token-consumption numbers
            # and break resume-on-interrupt for Colab runs.
            self.config = config
            self._backend = "llama_cpp"
            self._tracker = get_usage_tracker()
            self._lock = threading.Lock()
            self._cache_path = os.path.join(config.data_dir, "llm_cache.sqlite")
            self._init_cache()
        else:
            super().__init__(config)
        self._json_response_format_ok = False
        self._llama = None
        self._llama_lock = threading.Lock()
        if self._direct_llama:
            self._llama = self._load_llama_cpp()

    def _load_llama_cpp(self):
        try:
            from llama_cpp import Llama
        except ImportError as e:  # pragma: no cover - depends on Colab install
            raise RuntimeError(
                "PROPRAG_LLM_BACKEND=llama_cpp requires llama-cpp-python. "
                "Install it in Colab before running the benchmark."
            ) from e

        model_path = os.environ.get("PROPRAG_GGUF_MODEL_PATH", "")
        if not model_path or not os.path.isfile(model_path):
            raise RuntimeError(
                "PROPRAG_GGUF_MODEL_PATH must point to the downloaded GGUF file."
            )

        n_gpu_layers = _env_int("PROPRAG_LLAMA_N_GPU_LAYERS", -1)
        n_ctx = _env_int("PROPRAG_LLAMA_N_CTX", 8192)
        n_batch = _env_int("PROPRAG_LLAMA_N_BATCH", 512)
        n_threads = _env_int("PROPRAG_LLAMA_N_THREADS", 2)
        logger.info("Loading GGUF with llama.cpp: %s", model_path)
        return Llama(
            model_path=model_path,
            n_gpu_layers=n_gpu_layers,
            n_ctx=n_ctx,
            n_batch=n_batch,
            n_threads=n_threads,
            verbose=False,
        )

    def infer(self, messages: List[Dict[str, str]], *args, **kwargs):
        if not self._direct_llama:
            content, meta, cache_hit = super().infer(messages, *args, **kwargs)
            if self.config.strip_reasoning:
                content = strip_gpt_oss_reasoning(content)
            return content, meta, cache_hit
        return self._infer_direct(messages, *args, **kwargs)

    def unload(self):
        """Free the GGUF model from VRAM so the embedding model can load next.

        llama-cpp-python releases the model on ``__del__``; drop our reference and
        force GC + a CUDA cache flush so the freed VRAM is visible to the next
        phase. No-op for the HTTP (non-direct) backend.
        """
        import gc

        llama = getattr(self, "_llama", None)
        if llama is not None:
            close = getattr(llama, "close", None)
            if callable(close):
                try:
                    close()
                except Exception:  # noqa: BLE001
                    pass
        self._llama = None
        gc.collect()
        try:
            import torch

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:  # noqa: BLE001
            pass
        logger.info("GGUF chat model unloaded; VRAM freed.")

    def _infer_direct(
        self,
        messages: List[Dict[str, str]],
        temperature: float = None,
        json_mode: bool = False,
        use_cache: bool = True,
        max_completion_tokens: int = None,
        response_checker=None,
    ):
        temperature = self.config.temperature if temperature is None else temperature
        max_tokens = max_completion_tokens or self.config.max_completion_tokens
        # `system=None, turns=messages`: the exact split doesn't matter here, this
        # is purely a cache-key input, and "backend" already disambiguates it from
        # the HTTP cache path's keys.
        key = self._cache_key(None, messages, temperature, max_tokens, json_mode)

        if use_cache:
            cached = self._cache_get(key)
            if cached is not None:
                self._tracker.record(
                    CallRecord(kind="chat", model=_LLAMA_CPP_MODEL_LABEL, cache_hit=True)
                )
                content = strip_gpt_oss_reasoning(cached) if self.config.strip_reasoning else cached
                return content, {"finish_reason": "cached"}, True

        t0 = time.monotonic()
        with self._llama_lock:
            response = self._llama.create_chat_completion(
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                seed=self.config.seed if self.config.seed is not None else -1,
            )
        latency = time.monotonic() - t0
        choice = (response.get("choices") or [{}])[0]
        raw_content = (choice.get("message") or {}).get("content") or choice.get("text") or ""
        if not raw_content:
            raise RuntimeError("empty response from local llama.cpp model")

        usage = response.get("usage") or {}
        self._tracker.record(
            CallRecord(
                kind="chat",
                model=_LLAMA_CPP_MODEL_LABEL,
                prompt_tokens=int(usage.get("prompt_tokens", 0)),
                completion_tokens=int(usage.get("completion_tokens", 0)),
                latency_s=latency,
            )
        )
        if use_cache:
            self._cache_put(key, raw_content)

        content = strip_gpt_oss_reasoning(raw_content) if self.config.strip_reasoning else raw_content
        return content, {"finish_reason": choice.get("finish_reason", "stop")}, False


def check_backend(poc_cfg: POCConfig, timeout: float = 5.0) -> None:
    """Validate whichever LLM backend is configured."""
    if poc_cfg.llm_backend == "llama_cpp":
        model_path = os.environ.get("PROPRAG_GGUF_MODEL_PATH", "")
        if not model_path or not os.path.isfile(model_path):
            raise RuntimeError(
                "Cannot find the GGUF model. Set PROPRAG_GGUF_MODEL_PATH after "
                "downloading gpt-oss-20b-Q4_K_M.gguf from Hugging Face."
            )
        logger.info("Direct llama.cpp backend ready: %s", model_path)
        return

    base = (poc_cfg.llm_base_url or "").rstrip("/")
    url = f"{base}/models"
    hint = (
        "Cannot reach the LLM backend at "
        f"{poc_cfg.llm_base_url!r}. Start Koboldcpp first:\n"
        "  koboldcpp.exe --model gpt-oss-20b-Q4_K_M.gguf --port 5001 --contextsize 8192"
    )
    try:
        with urllib.request.urlopen(url, timeout=timeout) as resp:
            if resp.status >= 400:
                raise RuntimeError(f"{url} returned HTTP {resp.status}")
    except (urllib.error.URLError, OSError, RuntimeError) as e:
        raise RuntimeError(f"{hint}\n(underlying error: {e})") from e
    logger.info("LLM backend reachable at %s", poc_cfg.llm_base_url)


In [ ]:
%%writefile /content/benchmark/graphrag/__init__.py
"""Faithful Microsoft-style GraphRAG (own extraction, Leiden communities, reports).

Built new so its indexing cost is genuinely measured, rather than reusing the
PropRAG graph. Local search only.
"""


In [ ]:
%%writefile /content/benchmark/graphrag/extract.py
"""Typed entity + relationship extraction with a multi-tier JSON-repair chain.

Mirrors the repair strategy proven in ``proprag_poc/core/extraction.py``:
json_mode -> lenient parse -> ``fix_json`` repair call -> regex object salvage ->
empty result with a counted warning. Extraction workers set the
``index::GraphRAG`` usage scope so token attribution survives the thread pool.
"""

from __future__ import annotations

import json
import logging
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from typing import Dict, List, Set, Tuple

from proprag_poc.llm import prompts as poc_prompts
from . import prompts

logger = logging.getLogger(__name__)

_ENTITY_OBJ = re.compile(
    r'\{\s*"name"\s*:\s*"(?P<name>(?:\\.|[^"\\])*)"\s*,\s*"type"\s*:\s*'
    r'"(?P<type>(?:\\.|[^"\\])*)"\s*,\s*"description"\s*:\s*'
    r'"(?P<desc>(?:\\.|[^"\\])*)"',
    re.DOTALL,
)
_REL_OBJ = re.compile(
    r'\{\s*"source"\s*:\s*"(?P<src>(?:\\.|[^"\\])*)"\s*,\s*"target"\s*:\s*'
    r'"(?P<dst>(?:\\.|[^"\\])*)"\s*,\s*"description"\s*:\s*'
    r'"(?P<desc>(?:\\.|[^"\\])*)"(?:\s*,\s*"strength"\s*:\s*(?P<strength>\d+))?',
    re.DOTALL,
)


@dataclass
class EntityRec:
    key: str
    name: str
    type: str
    descriptions: List[str] = field(default_factory=list)
    chunk_ids: Set[str] = field(default_factory=set)
    summary: str = ""

    def merged_description(self) -> str:
        return self.summary or " ".join(dict.fromkeys(self.descriptions))


@dataclass
class RelRec:
    src_key: str
    dst_key: str
    descriptions: List[str] = field(default_factory=list)
    weight: float = 0.0
    chunk_ids: Set[str] = field(default_factory=set)


def _loads_lenient(raw: str):
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    return json.loads(s)


def _normalize(obj) -> Dict[str, list]:
    entities, rels = [], []
    if isinstance(obj, dict):
        for e in obj.get("entities", []) or []:
            if isinstance(e, dict) and e.get("name"):
                entities.append(e)
        for r in obj.get("relationships", []) or []:
            if isinstance(r, dict) and r.get("source") and r.get("target"):
                rels.append(r)
    return {"entities": entities, "relationships": rels}


def parse_extraction(raw: str) -> Dict[str, list]:
    """Lenient JSON parse; fall back to per-object regex salvage."""
    try:
        parsed = _normalize(_loads_lenient(raw))
        if parsed["entities"] or parsed["relationships"]:
            return parsed
    except Exception:  # noqa: BLE001
        pass

    entities = [
        {"name": m.group("name"), "type": m.group("type"), "description": m.group("desc")}
        for m in _ENTITY_OBJ.finditer(raw)
    ]
    rels = []
    for m in _REL_OBJ.finditer(raw):
        rels.append({
            "source": m.group("src"),
            "target": m.group("dst"),
            "description": m.group("desc"),
            "strength": int(m.group("strength")) if m.group("strength") else 5,
        })
    return {"entities": entities, "relationships": rels}


def extract_chunk(llm, text: str, cfg) -> Tuple[Dict[str, list], bool]:
    """Return (parsed, failed). ``failed`` is True only when nothing was salvaged."""
    raw, _, _ = llm.infer(
        prompts.extraction_messages(text, list(cfg.gr_entity_types)),
        json_mode=True,
        max_completion_tokens=cfg.extract_max_tokens,
    )
    parsed = parse_extraction(raw)
    if not parsed["entities"] and not parsed["relationships"]:
        fixed, _, _ = llm.infer(poc_prompts.fix_json_messages(raw), json_mode=True)
        parsed = parse_extraction(fixed)

    failed = not parsed["entities"] and not parsed["relationships"]

    for _ in range(max(0, cfg.gr_max_gleanings)):
        prev = json.dumps(parsed)
        graw, _, _ = llm.infer(
            prompts.gleaning_messages(text, prev, list(cfg.gr_entity_types)),
            json_mode=True,
            max_completion_tokens=cfg.extract_max_tokens,
        )
        extra = parse_extraction(graw)
        parsed["entities"].extend(extra["entities"])
        parsed["relationships"].extend(extra["relationships"])

    return parsed, failed


def _ekey(name: str) -> str:
    return name.strip().lower()


def batch_extract(
    llm, chunk_texts: Dict[str, str], cfg, tracker
) -> Tuple[Dict[str, EntityRec], List[RelRec], int]:
    """Parallel extraction + merge. Returns (entities, relationships, n_failures)."""
    keys = list(chunk_texts.keys())
    max_workers = llm.config.llm_max_workers
    logger.info("GraphRAG extract: %d chunks, %d workers", len(keys), max_workers)

    def _work(cid: str) -> Tuple[str, Dict[str, list], bool]:
        with tracker.scope("index::GraphRAG"):
            parsed, failed = extract_chunk(llm, chunk_texts[cid], cfg)
        return cid, parsed, failed

    per_chunk: Dict[str, Dict[str, list]] = {}
    n_failures = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futs = {ex.submit(_work, cid): cid for cid in keys}
        for done, fut in enumerate(as_completed(futs), 1):
            cid, parsed, failed = fut.result()
            per_chunk[cid] = parsed
            n_failures += int(failed)
            logger.info("GraphRAG extract %d/%d chunks done", done, len(keys))

    entities: Dict[str, EntityRec] = {}
    relationships: Dict[Tuple[str, str], RelRec] = {}

    def _ensure_entity(name: str, etype: str = "other") -> EntityRec:
        k = _ekey(name)
        rec = entities.get(k)
        if rec is None:
            rec = EntityRec(key=k, name=name.strip(), type=etype or "other")
            entities[k] = rec
        elif not rec.type or rec.type == "other":
            rec.type = etype or rec.type
        return rec

    for cid, parsed in per_chunk.items():
        for e in parsed["entities"]:
            rec = _ensure_entity(e["name"], e.get("type", "other"))
            desc = (e.get("description") or "").strip()
            if desc:
                rec.descriptions.append(desc)
            rec.chunk_ids.add(cid)
        for r in parsed["relationships"]:
            src = _ensure_entity(r["source"])          # stub endpoints (Microsoft behavior)
            dst = _ensure_entity(r["target"])
            if src.key == dst.key:
                continue
            pair = tuple(sorted((src.key, dst.key)))
            rel = relationships.get(pair)
            if rel is None:
                rel = RelRec(src_key=pair[0], dst_key=pair[1])
                relationships[pair] = rel
            desc = (r.get("description") or "").strip()
            if desc:
                rel.descriptions.append(desc)
            try:
                rel.weight += float(r.get("strength", 5) or 5)
            except (TypeError, ValueError):
                rel.weight += 5.0
            rel.chunk_ids.add(cid)

    logger.info("GraphRAG merged: %d entities, %d relationships (%d parse failures)",
                len(entities), len(relationships), n_failures)
    return entities, list(relationships.values()), n_failures


def summarize_entities(llm, entities: Dict[str, EntityRec], cfg) -> None:
    """LLM-merge descriptions only when their joined length exceeds the threshold."""
    n = 0
    for rec in entities.values():
        joined = " ".join(dict.fromkeys(rec.descriptions))
        if len(rec.descriptions) > 1 and len(joined) > cfg.gr_summarize_desc_threshold:
            raw, _, _ = llm.infer(
                prompts.summarize_descriptions_messages(rec.name, rec.descriptions),
                json_mode=True,
                max_completion_tokens=cfg.summarize_max_tokens,
            )
            try:
                rec.summary = (_loads_lenient(raw).get("description") or "").strip()
            except Exception:  # noqa: BLE001
                rec.summary = joined[: cfg.gr_summarize_desc_threshold]
            n += 1
    logger.info("GraphRAG summarized %d oversized entity descriptions", n)


In [ ]:
%%writefile /content/benchmark/graphrag/communities.py
"""Leiden community detection (igraph built-in, no leidenalg) + LLM reports."""

from __future__ import annotations

import json
import logging
import re
from typing import Dict, List

import igraph as ig

from .extract import EntityRec, RelRec
from . import prompts

logger = logging.getLogger(__name__)


def build_graph(entities: Dict[str, EntityRec], rels: List[RelRec]) -> ig.Graph:
    names = list(entities.keys())
    idx = {n: i for i, n in enumerate(names)}
    g = ig.Graph()
    g.add_vertices(len(names))
    g.vs["name"] = names

    edges, weights = [], []
    for rel in rels:
        if rel.src_key in idx and rel.dst_key in idx and rel.src_key != rel.dst_key:
            edges.append((idx[rel.src_key], idx[rel.dst_key]))
            weights.append(float(rel.weight) if rel.weight > 0 else 1.0)
    g.add_edges(edges)
    g.es["weight"] = weights
    logger.info("GraphRAG entity graph: %d nodes, %d edges", g.vcount(), g.ecount())
    return g


def detect_communities(g: ig.Graph) -> Dict[str, int]:
    """entity key -> community id via weighted-modularity Leiden."""
    if g.vcount() == 0:
        return {}
    if g.ecount() == 0:
        return {g.vs[i]["name"]: i for i in range(g.vcount())}
    clustering = g.community_leiden(
        objective_function="modularity", weights=g.es["weight"], n_iterations=5
    )
    membership = clustering.membership
    n = len(set(membership))
    logger.info("GraphRAG detected %d communities over %d entities", n, g.vcount())
    return {g.vs[i]["name"]: membership[i] for i in range(g.vcount())}


def _loads_lenient(raw: str):
    s = raw.strip()
    s = re.sub(r"^```(?:json)?", "", s).strip()
    s = re.sub(r"```$", "", s).strip()
    return json.loads(s)


def community_report(
    llm,
    member_keys: List[str],
    entities: Dict[str, EntityRec],
    intra_rels: List[RelRec],
    cfg,
) -> Dict:
    """Generate a report for one community; degrade to a minimal report on failure."""
    ent_lines = []
    for k in member_keys:
        rec = entities.get(k)
        if rec is None:
            continue
        ent_lines.append(f"{rec.name} ({rec.type}): {rec.merged_description()}")
    entities_block = "\n".join(ent_lines)

    rel_lines = [
        f"{entities[r.src_key].name} -> {entities[r.dst_key].name}: "
        f"{' ; '.join(dict.fromkeys(r.descriptions)) or 'related'} (w={r.weight:.0f})"
        for r in sorted(intra_rels, key=lambda x: x.weight, reverse=True)
        if r.src_key in entities and r.dst_key in entities
    ]
    rels_block = "\n".join(rel_lines)

    # Truncate the combined context to the configured character budget.
    budget = cfg.gr_report_max_input_chars
    if len(entities_block) + len(rels_block) > budget:
        entities_block = entities_block[: budget // 2]
        rels_block = rels_block[: budget // 2]

    try:
        raw, _, _ = llm.infer(
            prompts.community_report_messages(entities_block, rels_block),
            json_mode=True,
            max_completion_tokens=cfg.report_max_tokens,
        )
        report = _loads_lenient(raw)
        if not isinstance(report, dict):
            raise ValueError("report is not a JSON object")
        report.setdefault("title", f"Community of {member_keys[0] if member_keys else '?'}")
        report.setdefault("summary", "")
        report.setdefault("rating", 0)
        report.setdefault("findings", [])
        return report
    except Exception as e:  # noqa: BLE001
        logger.warning("community report failed, using stub: %s", e)
        titles = ", ".join(entities[k].name for k in member_keys[:3] if k in entities)
        return {
            "title": titles or "Community",
            "summary": entities_block[:400],
            "rating": 0,
            "findings": [],
            "parse_failed": True,
        }


In [ ]:
%%writefile /content/benchmark/graphrag/prompts.py
"""GraphRAG prompts: entity/relationship extraction, description summarization,
and community reports. All emit JSON (``json_mode=True``), kept short with one
worked example so a CPU-served 20B model stays parseable.
"""

from __future__ import annotations

from typing import Dict, List

# ------------------------------------------------------------------ extraction
_EXTRACT_SYSTEM = """You are a knowledge-graph extractor. From the text, extract
named entities and the relationships between them.

Return ONLY a JSON object with this exact shape:
{{"entities": [{{"name": "...", "type": "...", "description": "..."}}],
 "relationships": [{{"source": "...", "target": "...", "description": "...", "strength": 1-10}}]}}

Rules:
- "type" must be one of: {entity_types}.
- "name" is the entity as it appears (proper noun / phrase).
- "description" is a short factual phrase grounded in the text.
- "source" and "target" of every relationship must be entity names you also list.
- "strength" is an integer 1-10 rating how strongly the text ties the two entities.
Extract nothing that is not supported by the text."""

_EXTRACT_EXAMPLE = """Example.
Text: "Marie Curie, a physicist, was born in Warsaw and won the Nobel Prize."
JSON:
{"entities": [
  {"name": "Marie Curie", "type": "person", "description": "physicist born in Warsaw"},
  {"name": "Warsaw", "type": "location", "description": "birthplace of Marie Curie"},
  {"name": "Nobel Prize", "type": "work", "description": "award won by Marie Curie"}],
 "relationships": [
  {"source": "Marie Curie", "target": "Warsaw", "description": "was born in", "strength": 8},
  {"source": "Marie Curie", "target": "Nobel Prize", "description": "won the", "strength": 9}]}"""

_EXTRACT_USER = """{example}

Now extract from this text.
Text:
```
{text}
```
JSON:"""


def extraction_messages(text: str, entity_types: List[str]) -> List[Dict[str, str]]:
    system = _EXTRACT_SYSTEM.format(entity_types=", ".join(entity_types))
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": _EXTRACT_USER.format(example=_EXTRACT_EXAMPLE, text=text)},
    ]


_GLEAN_USER = """Some entities and relationships were missed in the previous
extraction of the same text. Add ONLY new ones, using the exact same JSON shape
{{"entities": [...], "relationships": [...]}}. Do not repeat anything already
listed below.

Text:
```
{text}
```

Already extracted:
{prev_json}

Additional JSON:"""


def gleaning_messages(text: str, prev_json: str, entity_types: List[str]) -> List[Dict[str, str]]:
    system = _EXTRACT_SYSTEM.format(entity_types=", ".join(entity_types))
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": _GLEAN_USER.format(text=text, prev_json=prev_json)},
    ]


# ------------------------------------------------------- description summaries
_SUMMARIZE_SYSTEM = """You merge several descriptions of the same entity into one
concise, factual description. Return ONLY JSON: {"description": "..."}."""

_SUMMARIZE_USER = """Entity: {name}
Descriptions:
{descriptions}

Merged description as JSON:"""


def summarize_descriptions_messages(name: str, descriptions: List[str]) -> List[Dict[str, str]]:
    body = "\n".join(f"- {d}" for d in descriptions)
    return [
        {"role": "system", "content": _SUMMARIZE_SYSTEM},
        {"role": "user", "content": _SUMMARIZE_USER.format(name=name, descriptions=body)},
    ]


# ----------------------------------------------------------- community reports
_REPORT_SYSTEM = """You write a concise analytical report about a community of
related entities. Return ONLY JSON:
{"title": "...", "summary": "...", "rating": 0-10,
 "findings": [{"summary": "...", "explanation": "..."}]}
"rating" is the community's overall importance. Keep the summary under 120 words
and base everything strictly on the provided entities and relationships."""

_REPORT_USER = """Entities:
{entities_block}

Relationships:
{rels_block}

Write the community report as JSON:"""


def community_report_messages(entities_block: str, rels_block: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": _REPORT_SYSTEM},
        {"role": "user", "content": _REPORT_USER.format(
            entities_block=entities_block, rels_block=rels_block)},
    ]


In [ ]:
%%writefile /content/benchmark/graphrag/search.py
"""GraphRAG LOCAL search: query -> entity match (by embedding) -> chunk scoring.

Retrieval-comparable with the other systems (Recall@k is computed from
``retrieve()`` which returns documents only). ``build_qa_context`` assembles the
faithful local-search context - community reports + relationship lines + chunks -
within the same ``qa_top_k`` budget used by BaseRAG/PropRAG.
"""

from __future__ import annotations

import logging
from typing import Dict, List, Tuple

import numpy as np

from proprag_poc.core.retriever import RetrievedPassage, _min_max

from .index import GraphRAGIndex

logger = logging.getLogger(__name__)


class GraphRAGLocalRetriever:
    system_name = "GraphRAG"

    def __init__(self, index: GraphRAGIndex, embedding_model, poc_cfg, bench_cfg):
        self.index = index
        self.embedding_model = embedding_model
        self.poc_cfg = poc_cfg
        self.cfg = bench_cfg

        self.passage_keys = index.chunk_store.get_all_ids()
        self.passage_row = {k: i for i, k in enumerate(self.passage_keys)}
        self.passage_embeddings = index.chunk_store.get_embeddings(self.passage_keys)

        self.entity_row = {k: i for i, k in enumerate(index.entity_order)}
        self.rel_lookup: Dict[frozenset, object] = {
            frozenset((r.src_key, r.dst_key)): r for r in index.relationships
        }

    # ------------------------------------------------------------- helpers
    def _embed_query(self, query: str) -> np.ndarray:
        return self.embedding_model.batch_encode(
            query, instruction=self.poc_cfg.embedding_query_instruction,
            norm=True, input_type="query",
        ).squeeze()

    def _match_entities(self, q: np.ndarray) -> List[Tuple[str, float]]:
        if self.index.entity_embeddings.shape[0] == 0 or self.index.entity_embeddings.ndim != 2:
            return []
        sims = self.index.entity_embeddings @ q.T
        order = np.argsort(sims)[::-1][: self.cfg.gr_local_top_entities]
        out = []
        for i in order:
            score = float(sims[i])
            if score < self.cfg.gr_entity_sim_floor:
                break
            out.append((self.index.entity_order[i], score))
        return out

    def _dense_scores(self, q: np.ndarray) -> np.ndarray:
        return np.atleast_1d((self.passage_embeddings @ q.T).squeeze())

    # ------------------------------------------------------------- retrieve
    def retrieve(self, query: str, top_k: int = None) -> List[RetrievedPassage]:
        top_k = top_k or self.poc_cfg.retrieval_top_k
        q = self._embed_query(query)
        dense = _min_max(self._dense_scores(q))
        matched = self._match_entities(q)

        if not matched:
            order = np.argsort(dense)[::-1][:top_k]
            return [self._passage(i, float(dense[i])) for i in order]

        graph_score = np.zeros(len(self.passage_keys))
        for ekey, sim in matched:
            for cid in self.index.entity_chunks.get(ekey, []):
                row = self.passage_row.get(cid)
                if row is not None:
                    graph_score[row] += sim

        # Bonus for chunks cited by a relationship between two matched entities.
        matched_keys = [k for k, _ in matched]
        for a in range(len(matched_keys)):
            for b in range(a + 1, len(matched_keys)):
                rel = self.rel_lookup.get(frozenset((matched_keys[a], matched_keys[b])))
                if rel is None:
                    continue
                for cid in rel.chunk_ids:
                    row = self.passage_row.get(cid)
                    if row is not None:
                        graph_score[row] += 0.5

        if graph_score.sum() == 0:
            order = np.argsort(dense)[::-1][:top_k]
            return [self._passage(i, float(dense[i])) for i in order]

        final = _min_max(graph_score) + self.cfg.gr_dense_blend * dense
        order = np.argsort(final)[::-1][:top_k]
        return [self._passage(i, float(final[i])) for i in order]

    def _passage(self, row: int, score: float) -> RetrievedPassage:
        key = self.passage_keys[row]
        return RetrievedPassage(
            chunk_id=key,
            text=self.index.chunk_store.get_row(key)["content"],
            score=score,
        )

    # ---------------------------------------------------------- QA context
    def build_qa_context(self, query: str, passages: List[RetrievedPassage]) -> List[str]:
        """Local-search context within the qa_top_k budget: community reports +
        relationship lines + top chunks."""
        budget = self.cfg.qa_top_k
        q = self._embed_query(query)
        matched = self._match_entities(q)
        matched_keys = [k for k, _ in matched]

        context: List[str] = []

        # 1-2 community reports covering the matched entities (highest rating first).
        seen_comm = set()
        comm_reports = []
        for ekey in matched_keys:
            cid = self.index.communities.get(ekey)
            if cid is None or cid in seen_comm:
                continue
            report = self.index.reports.get(int(cid))
            if report:
                seen_comm.add(cid)
                comm_reports.append(report)
        comm_reports.sort(key=lambda r: r.get("rating", 0), reverse=True)
        for report in comm_reports[:2]:
            context.append(f"[Community report] {report.get('title', '')}: {report.get('summary', '')}")

        # Relationship one-liners between matched entities.
        rel_lines = []
        for a in range(len(matched_keys)):
            for b in range(a + 1, len(matched_keys)):
                rel = self.rel_lookup.get(frozenset((matched_keys[a], matched_keys[b])))
                if rel is None:
                    continue
                na = self.index.entities[rel.src_key].name
                nb = self.index.entities[rel.dst_key].name
                desc = " ; ".join(dict.fromkeys(rel.descriptions)) or "related"
                rel_lines.append(f"{na} -> {nb}: {desc}")
        if rel_lines:
            context.append("[Relationships]\n" + "\n".join(rel_lines[:10]))

        # Fill the remaining budget with top retrieved chunks.
        for p in passages:
            if len(context) >= budget:
                break
            context.append(p.text)
        return context[:budget] if context else [p.text for p in passages[:budget]]


In [ ]:
%%writefile /content/benchmark/graphrag/index.py
"""GraphRAG index: build / persist / load.

Reuses the SHARED chunk embedding store (no re-embedding of passages). Persists
under ``data/corpora/<corpus_id>/graphrag/``:
  extraction.json   - entities + relationships checkpoint (after batch_extract)
  entities.json     - merged/summarized entities (with embed text + provenance)
  relationships.json, communities.json, reports.json
  graph.graphml     - entity graph
  gr_entity_*       - EmbeddingStore for "name: description" vectors
"""

from __future__ import annotations

import json
import logging
import os
from dataclasses import dataclass, field
from typing import Dict, List

import igraph as ig
import numpy as np

from proprag_poc.core.ids import compute_mdhash_id
from proprag_poc.core.store import EmbeddingStore
from proprag_poc.embedding.encoder import EmbeddingModel

from . import communities as comm
from .extract import EntityRec, RelRec, batch_extract, summarize_entities

logger = logging.getLogger(__name__)

_GR_NAMESPACE = "gr_entity"


def _embed_text(rec: EntityRec) -> str:
    return f"{rec.name}: {rec.merged_description()}".strip()[:512]


# ------------------------------------------------------------- (de)serialize
def _entity_to_dict(rec: EntityRec) -> Dict:
    return {
        "key": rec.key,
        "name": rec.name,
        "type": rec.type,
        "descriptions": rec.descriptions,
        "chunk_ids": sorted(rec.chunk_ids),
        "summary": rec.summary,
    }


def _entity_from_dict(d: Dict) -> EntityRec:
    return EntityRec(
        key=d["key"], name=d["name"], type=d["type"],
        descriptions=list(d.get("descriptions", [])),
        chunk_ids=set(d.get("chunk_ids", [])),
        summary=d.get("summary", ""),
    )


def _rel_to_dict(r: RelRec) -> Dict:
    return {
        "src_key": r.src_key, "dst_key": r.dst_key,
        "descriptions": r.descriptions, "weight": r.weight,
        "chunk_ids": sorted(r.chunk_ids),
    }


def _rel_from_dict(d: Dict) -> RelRec:
    return RelRec(
        src_key=d["src_key"], dst_key=d["dst_key"],
        descriptions=list(d.get("descriptions", [])),
        weight=float(d.get("weight", 0.0)),
        chunk_ids=set(d.get("chunk_ids", [])),
    )


@dataclass
class GraphRAGIndex:
    entities: Dict[str, EntityRec]
    relationships: List[RelRec]
    communities: Dict[str, int]          # entity key -> community id
    reports: Dict[int, Dict]             # community id -> report
    entity_order: List[str]              # entity keys aligned to entity_embeddings
    entity_embeddings: np.ndarray
    entity_store: EmbeddingStore
    graph: ig.Graph
    chunk_store: EmbeddingStore          # SHARED
    entity_chunks: Dict[str, List[str]]  # entity key -> chunk ids
    n_extract_failures: int = 0


def _gdir(corpus_dir: str) -> str:
    return os.path.join(corpus_dir, "graphrag")


def _paths(gdir: str) -> Dict[str, str]:
    return {name: os.path.join(gdir, f"{name}.json") for name in
            ("extraction", "entities", "relationships", "communities", "reports")} | {
        "graph": os.path.join(gdir, "graph.graphml"),
    }


def _exists(gdir: str) -> bool:
    p = _paths(gdir)
    return all(os.path.isfile(p[k]) for k in ("entities", "relationships", "communities", "reports", "graph"))


def _fetch_embeddings(store: EmbeddingStore, texts: List[str]) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1), dtype=np.float32)
    ids = [compute_mdhash_id(t, prefix=f"{_GR_NAMESPACE}-") for t in texts]
    return store.get_embeddings(ids)


def build_or_load(
    poc_cfg, bench_cfg, corpus_dir: str, chunk_store: EmbeddingStore,
    emb: EmbeddingModel, llm, tracker, force: bool = False,
) -> GraphRAGIndex:
    gdir = _gdir(corpus_dir)
    os.makedirs(gdir, exist_ok=True)
    p = _paths(gdir)
    entity_store = EmbeddingStore(emb, gdir, _GR_NAMESPACE)

    if _exists(gdir) and not force:
        logger.info("Loading existing GraphRAG index")
        return _load(gdir, entity_store, chunk_store)

    # --- extraction (checkpointed) ---
    n_failures = 0
    if os.path.isfile(p["extraction"]) and not force:
        logger.info("Loading GraphRAG extraction checkpoint")
        with open(p["extraction"], "r", encoding="utf-8") as f:
            ck = json.load(f)
        entities = {k: _entity_from_dict(v) for k, v in ck["entities"].items()}
        relationships = [_rel_from_dict(r) for r in ck["relationships"]]
        n_failures = ck.get("n_extract_failures", 0)
    else:
        chunk_texts = {
            cid: row["content"] for cid, row in chunk_store.get_text_for_all_rows().items()
        }
        entities, relationships, n_failures = batch_extract(llm, chunk_texts, bench_cfg, tracker)
        summarize_entities(llm, entities, bench_cfg)
        with open(p["extraction"], "w", encoding="utf-8") as f:
            json.dump({
                "entities": {k: _entity_to_dict(v) for k, v in entities.items()},
                "relationships": [_rel_to_dict(r) for r in relationships],
                "n_extract_failures": n_failures,
            }, f)

    # --- entity graph + communities ---
    graph = comm.build_graph(entities, relationships)
    community_map = comm.detect_communities(graph)

    members: Dict[int, List[str]] = {}
    for key, cid in community_map.items():
        members.setdefault(cid, []).append(key)

    rels_by_comm: Dict[int, List[RelRec]] = {}
    for r in relationships:
        c1, c2 = community_map.get(r.src_key), community_map.get(r.dst_key)
        if c1 is not None and c1 == c2:
            rels_by_comm.setdefault(c1, []).append(r)

    reports: Dict[int, Dict] = {}
    for cid, keys in members.items():
        if len(keys) < bench_cfg.gr_min_community_size:
            continue
        reports[cid] = comm.community_report(
            llm, keys, entities, rels_by_comm.get(cid, []), bench_cfg
        )
    logger.info("GraphRAG built %d community reports", len(reports))

    # --- entity embeddings ("name: description") ---
    entity_order = list(entities.keys())
    embed_texts = [_embed_text(entities[k]) for k in entity_order]
    entity_store.insert_strings(embed_texts)
    entity_embeddings = _fetch_embeddings(entity_store, embed_texts)

    entity_chunks = {k: sorted(entities[k].chunk_ids) for k in entity_order}

    _persist(gdir, entities, relationships, community_map, reports,
             entity_order, embed_texts, entity_chunks, graph, n_failures)

    return GraphRAGIndex(
        entities=entities, relationships=relationships, communities=community_map,
        reports=reports, entity_order=entity_order, entity_embeddings=entity_embeddings,
        entity_store=entity_store, graph=graph, chunk_store=chunk_store,
        entity_chunks=entity_chunks, n_extract_failures=n_failures,
    )


def _persist(gdir, entities, relationships, community_map, reports, entity_order,
             embed_texts, entity_chunks, graph, n_failures) -> None:
    p = _paths(gdir)
    with open(p["entities"], "w", encoding="utf-8") as f:
        json.dump({
            "entities": {k: _entity_to_dict(v) for k, v in entities.items()},
            "entity_order": entity_order,
            "embed_texts": embed_texts,
            "entity_chunks": entity_chunks,
            "n_extract_failures": n_failures,
        }, f)
    with open(p["relationships"], "w", encoding="utf-8") as f:
        json.dump([_rel_to_dict(r) for r in relationships], f)
    with open(p["communities"], "w", encoding="utf-8") as f:
        json.dump(community_map, f)
    with open(p["reports"], "w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in reports.items()}, f)
    graph.write_graphml(p["graph"])


def _load(gdir, entity_store, chunk_store) -> GraphRAGIndex:
    p = _paths(gdir)
    with open(p["entities"], "r", encoding="utf-8") as f:
        edata = json.load(f)
    entities = {k: _entity_from_dict(v) for k, v in edata["entities"].items()}
    entity_order = edata["entity_order"]
    embed_texts = edata["embed_texts"]
    entity_chunks = edata["entity_chunks"]
    n_failures = edata.get("n_extract_failures", 0)
    with open(p["relationships"], "r", encoding="utf-8") as f:
        relationships = [_rel_from_dict(r) for r in json.load(f)]
    with open(p["communities"], "r", encoding="utf-8") as f:
        community_map = json.load(f)
    with open(p["reports"], "r", encoding="utf-8") as f:
        reports = {int(k): v for k, v in json.load(f).items()}
    graph = ig.Graph.Read_GraphML(p["graph"])
    entity_embeddings = _fetch_embeddings(entity_store, embed_texts)
    return GraphRAGIndex(
        entities=entities, relationships=relationships, communities=community_map,
        reports=reports, entity_order=entity_order, entity_embeddings=entity_embeddings,
        entity_store=entity_store, graph=graph, chunk_store=chunk_store,
        entity_chunks=entity_chunks, n_extract_failures=n_failures,
    )


In [ ]:
%%writefile /content/benchmark/run_colab.py
"""Colab, VRAM-safe benchmark driver: PropRAG vs GraphRAG vs BaseRAG on 2wiki.

A free-tier T4 (15GB) cannot hold NV-Embed-v2 and the gpt-oss-20b GGUF at the
same time, so the interleaved desktop pipeline is split into three phases where
each model is resident at most once and the two are NEVER co-resident:

    Phase A  (chat LLM resident)  extraction only:
                 PropRAG NER + propositions, GraphRAG entities / relationships /
                 community reports. All text-in / text-out. -> unload LLM.

    Phase B  (embedder resident)  embeddings + retrieval:
                 build the shared chunk store + PropRAG/GraphRAG vector stores +
                 graphs, then run the FULL retrieval for every (question, system)
                 -- retrieval uses NO LLM -- and persist the retrieved passages,
                 the QA context and Recall@k. -> unload embedder.

    Phase C  (chat LLM resident)  QA only:
                 answer each question from its stored context, score EM/F1,
                 write results.jsonl and build the report.

Everything is written under ``data_dir`` (mount it on Google Drive) and every
phase is resume-safe: re-running skips work whose outputs already exist, so a
Colab disconnect never loses completed extraction / embedding / QA.

    python -m benchmark.run_colab --pilot 12 --phase all
    python -m benchmark.run_colab --phase a        # just extraction
    python -m benchmark.run_colab --report-only
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple

from . import _bootstrap  # noqa: F401 - side effect: proprag_poc on sys.path

import igraph as ig

from proprag_poc.logging_setup import setup_logging
from proprag_poc.core.metrics import get_usage_tracker
from proprag_poc.core.ids import compute_mdhash_id
from proprag_poc.core.store import EmbeddingStore
from proprag_poc.core.extraction import Extractor
from proprag_poc.core.graph_builder import GraphBuilder
from proprag_poc.core.retriever import Retriever
from proprag_poc.core.baseline_retrievers import BaseRAGRetriever
from proprag_poc.embedding.encoder import EmbeddingModel

from .bench_config import BenchmarkConfig
from .dataset import (
    build_corpus, chunk_text, corpus_id, load_questions, pilot_subset,
    stratified_subset, subset_hash, write_manifest,
)
from .evaluation import em_score, f1_score, recall_at_k
from .qa import answer_question
from .llm_wrap import BenchLLMClient, check_backend
from .results import ResultsStore
from . import report as report_mod
from .graphrag import extract as gr_extract
from .graphrag import communities as gr_comm
from .graphrag import index as gr_index
from .graphrag.search import GraphRAGLocalRetriever

logger = logging.getLogger("benchmark")


# --------------------------------------------------------------------- logging
def _setup_logging() -> None:
    setup_logging()
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-5s | %(name)s | %(message)s", "%H:%M:%S")
    )
    root = logging.getLogger("benchmark")
    root.setLevel(logging.INFO)
    root.handlers.clear()
    root.addHandler(handler)
    root.propagate = False


# ---------------------------------------------------------------- shared state
@dataclass
class BenchCorpus:
    """Duck-typed container satisfying the POC ``Retriever`` + ``BaseRAGRetriever``."""

    corpus_id: str
    chunk_store: EmbeddingStore
    entity_store: EmbeddingStore
    proposition_store: EmbeddingStore
    graph: ig.Graph
    proposition_to_entities_map: Dict[str, List[str]]
    chunk_propositions: Dict[str, List[Dict]]
    chunk_id_to_title: Dict[str, str]


@dataclass
class _Ctx:
    poc_cfg: object
    questions: list
    titles: Dict[str, str]
    docs: List[Tuple[str, str]]
    corpus_ident: str
    cid2text: Dict[str, str]
    cid2title: Dict[str, str]
    cdir: str
    run_dir: str
    systems: List[str]


def _run_id(cfg: BenchmarkConfig) -> str:
    rid = f"{cfg.n_questions}q_seed{cfg.seed}"
    return f"{rid}_pilot{cfg.pilot}" if cfg.pilot else rid


def _prepare(cfg: BenchmarkConfig) -> _Ctx:
    """Deterministic, model-free setup shared by every phase (cheap to recompute)."""
    poc_cfg = cfg.make_poc_config()
    all_qs = load_questions(cfg.dataset_path)
    subset = stratified_subset(all_qs, cfg.n_questions, cfg.seed)
    questions = pilot_subset(subset, cfg.pilot, cfg.seed) if cfg.pilot else subset

    titles = build_corpus(questions, cfg.corpus_path)
    docs = [(t, titles[t]) for t in sorted(titles)]
    tag = f"n{cfg.n_questions}" + (f"_pilot{cfg.pilot}" if cfg.pilot else "")
    corpus_ident = corpus_id(questions, tag)

    cid2text, cid2title = {}, {}
    for title, text in docs:
        ct = chunk_text(title, text)
        cid = compute_mdhash_id(ct, prefix="chunk-")
        cid2text[cid] = ct
        cid2title[cid] = title

    cdir = os.path.join(poc_cfg.data_dir, "corpora", corpus_ident)
    run_dir = os.path.join(poc_cfg.data_dir, "benchmark", _run_id(cfg))
    os.makedirs(cdir, exist_ok=True)
    os.makedirs(run_dir, exist_ok=True)
    systems = [s.strip() for s in cfg.systems if s.strip()]
    return _Ctx(poc_cfg, questions, titles, docs, corpus_ident, cid2text, cid2title,
                cdir, run_dir, systems)


# ------------------------------------------------------------------- io helpers
def _load_json(path: str, default):
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return default


def _save_json(path: str, obj) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def _read_jsonl(path: str) -> List[Dict]:
    out: List[Dict] = []
    if not os.path.isfile(path):
        return out
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                out.append(json.loads(line))
    return out


def _append_jsonl(path: str, row: Dict) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")
        f.flush()
        os.fsync(f.fileno())


def _done_keys(path: str) -> Set[Tuple[str, str]]:
    return {(r["qid"], r["system"]) for r in _read_jsonl(path)
            if "qid" in r and "system" in r}


def _delta(before: Dict, after: Dict) -> Dict:
    keys = set(before) | set(after)
    return {k: after.get(k, 0) - before.get(k, 0) for k in keys}


def _add(a: Dict, b: Dict) -> Dict:
    keys = set(a) | set(b)
    return {k: a.get(k, 0) + b.get(k, 0) for k in keys}


def _chat_fields(d: Dict) -> Dict:
    return {
        "chat_calls": d.get("chat_calls", 0),
        "chat_prompt_tokens": d.get("chat_prompt_tokens", 0),
        "chat_completion_tokens": d.get("chat_completion_tokens", 0),
    }


def _write_meta(ctx: _Ctx, cfg: BenchmarkConfig) -> None:
    meta = {
        "subset_hash": subset_hash(ctx.questions),
        "seed": cfg.seed,
        "n_questions": cfg.n_questions,
        "pilot": cfg.pilot,
        "llm_backend": ctx.poc_cfg.llm_backend,
        "llm_model": os.environ.get("PROPRAG_GGUF_LABEL", ctx.poc_cfg.llm_model),
        "embedding_model": ctx.poc_cfg.embedding_model,
        "qa_top_k": cfg.qa_top_k,
        "retrieval_top_k": cfg.retrieval_top_k,
        "gr_max_gleanings": cfg.gr_max_gleanings,
    }
    _save_json(os.path.join(ctx.run_dir, "run_meta.json"), meta)


# ====================================================================== PHASE A
def phase_a_extract(cfg: BenchmarkConfig, force: bool = False) -> str:
    """Chat-LLM-only extraction for all three systems. Unloads the LLM on exit."""
    ctx = _prepare(cfg)
    write_manifest(ctx.run_dir, ctx.questions, ctx.titles, cfg.seed, cfg)
    _write_meta(ctx, cfg)

    openie_path = os.path.join(ctx.cdir, "openie.json")
    gdir = gr_index._gdir(ctx.cdir)
    tracker = get_usage_tracker()
    chat_usage = _load_json(os.path.join(ctx.run_dir, "index_chat_usage.json"), {})

    need_prop = force or not os.path.isfile(openie_path)
    need_gr = force or not gr_index._exists(gdir)
    if not need_prop and not need_gr:
        logger.info("Phase A: extraction already complete; not loading the LLM.")
        return ctx.run_dir

    check_backend(ctx.poc_cfg)
    logger.info("Phase A: loading GGUF chat model for extraction ...")
    llm = BenchLLMClient(ctx.poc_cfg)
    try:
        if need_prop:
            logger.info("Phase A: PropRAG NER + proposition extraction (%d chunks)",
                        len(ctx.cid2text))
            before = tracker.snapshot("index").as_dict()
            t0 = time.monotonic()
            chunk_props = Extractor(llm).batch_extract(ctx.cid2text)
            _save_json(openie_path, chunk_props)
            d = _delta(before, tracker.snapshot("index").as_dict())
            chat_usage["PropRAG"] = {**_chat_fields(d), "wall_time_s": round(time.monotonic() - t0, 3)}

        if need_gr:
            logger.info("Phase A: GraphRAG extraction + community reports")
            b_idx = tracker.snapshot("index").as_dict()
            b_gr = tracker.snapshot("index::GraphRAG").as_dict()
            t0 = time.monotonic()
            n_failures = _build_graphrag_llm(llm, ctx.cid2text, cfg, gdir, tracker, force)
            d = _add(_delta(b_idx, tracker.snapshot("index").as_dict()),
                     _delta(b_gr, tracker.snapshot("index::GraphRAG").as_dict()))
            chat_usage["GraphRAG"] = {
                **_chat_fields(d),
                "wall_time_s": round(time.monotonic() - t0, 3),
                "parse_failures": n_failures,
            }

        _save_json(os.path.join(ctx.run_dir, "index_chat_usage.json"), chat_usage)
    finally:
        llm.unload()
    logger.info("Phase A complete.")
    return ctx.run_dir


def _build_graphrag_llm(llm, chunk_texts, cfg, gdir, tracker, force) -> int:
    """GraphRAG extraction + communities + reports (no embeddings). Persists JSON.

    Mirrors ``graphrag.index.build_or_load`` minus the entity-embedding step,
    which is deferred to Phase B. Returns the parse-failure count.
    """
    os.makedirs(gdir, exist_ok=True)
    p = gr_index._paths(gdir)

    if os.path.isfile(p["extraction"]) and not force:
        logger.info("Phase A: loading GraphRAG extraction checkpoint")
        with open(p["extraction"], "r", encoding="utf-8") as f:
            ck = json.load(f)
        entities = {k: gr_index._entity_from_dict(v) for k, v in ck["entities"].items()}
        relationships = [gr_index._rel_from_dict(r) for r in ck["relationships"]]
        n_failures = ck.get("n_extract_failures", 0)
    else:
        entities, relationships, n_failures = gr_extract.batch_extract(
            llm, chunk_texts, cfg, tracker
        )
        gr_extract.summarize_entities(llm, entities, cfg)
        with open(p["extraction"], "w", encoding="utf-8") as f:
            json.dump({
                "entities": {k: gr_index._entity_to_dict(v) for k, v in entities.items()},
                "relationships": [gr_index._rel_to_dict(r) for r in relationships],
                "n_extract_failures": n_failures,
            }, f)

    graph = gr_comm.build_graph(entities, relationships)
    community_map = gr_comm.detect_communities(graph)

    members: Dict[int, List[str]] = {}
    for key, cid in community_map.items():
        members.setdefault(cid, []).append(key)

    rels_by_comm: Dict[int, List] = {}
    for r in relationships:
        c1, c2 = community_map.get(r.src_key), community_map.get(r.dst_key)
        if c1 is not None and c1 == c2:
            rels_by_comm.setdefault(c1, []).append(r)

    reports: Dict[int, Dict] = {}
    for cid, keys in members.items():
        if len(keys) < cfg.gr_min_community_size:
            continue
        reports[cid] = gr_comm.community_report(
            llm, keys, entities, rels_by_comm.get(cid, []), cfg
        )
    logger.info("Phase A: built %d GraphRAG community reports", len(reports))

    entity_order = list(entities.keys())
    embed_texts = [gr_index._embed_text(entities[k]) for k in entity_order]
    entity_chunks = {k: sorted(entities[k].chunk_ids) for k in entity_order}
    gr_index._persist(gdir, entities, relationships, community_map, reports,
                      entity_order, embed_texts, entity_chunks, graph, n_failures)
    return n_failures


# ====================================================================== PHASE B
def phase_b_embed_retrieve(cfg: BenchmarkConfig, force: bool = False) -> str:
    """Embedder-only phase: build vector stores + graphs, run all retrieval.

    No chat LLM is loaded. Persists ``retrieval.jsonl`` (retrieved passages, QA
    context, Recall@k) and merges the final ``index_usage.json``. Unloads the
    embedder on exit.
    """
    ctx = _prepare(cfg)
    retr_path = os.path.join(ctx.run_dir, "retrieval.jsonl")
    done = _done_keys(retr_path)
    all_pairs = {(q.qid, s) for q in ctx.questions for s in ctx.systems}

    openie_path = os.path.join(ctx.cdir, "openie.json")
    if not os.path.isfile(openie_path):
        raise RuntimeError("Phase B needs Phase A output (openie.json). Run --phase a first.")

    if all_pairs.issubset(done) and not force:
        logger.info("Phase B: retrieval already complete; not loading the embedder.")
        _merge_index_usage(ctx, {})
        return ctx.run_dir

    logger.info("Phase B: loading embedder (8-bit NV-Embed on GPU) ...")
    emb = EmbeddingModel(ctx.poc_cfg)
    tracker = get_usage_tracker()
    embed_usage: Dict[str, Dict] = {}
    try:
        # --- BaseRAG index: shared chunk store ---
        chunk_store = EmbeddingStore(emb, ctx.cdir, "chunk")
        before = tracker.snapshot("index").as_dict()
        t0 = time.monotonic()
        chunk_store.insert_strings([chunk_text(t, x) for t, x in ctx.docs])
        _save_json(os.path.join(ctx.cdir, "title_map.json"), ctx.cid2title)
        embed_usage["BaseRAG"] = {
            "embed_texts": _delta(before, tracker.snapshot("index").as_dict()).get("embed_texts", 0),
            "wall_time_s": round(time.monotonic() - t0, 3),
        }

        # --- PropRAG index: entities + propositions + knowledge graph ---
        corpus = _build_proprag_corpus(ctx, emb, chunk_store, openie_path, tracker, embed_usage, force)

        # --- GraphRAG index: entity embeddings + load structure from Phase A ---
        gr = _load_graphrag_index(ctx, emb, chunk_store, tracker, embed_usage)

        retrievers = {
            "BaseRAG": BaseRAGRetriever(corpus, emb, ctx.poc_cfg),
            "PropRAG": Retriever(corpus, emb, ctx.poc_cfg),
            "GraphRAG": GraphRAGLocalRetriever(gr, emb, ctx.poc_cfg, cfg),
        }

        _retrieve_all(ctx, cfg, retrievers, retr_path, done)
        _save_json(os.path.join(ctx.run_dir, "index_embed_usage.json"), embed_usage)
        _merge_index_usage(ctx, embed_usage)
    finally:
        emb.unload()
    logger.info("Phase B complete.")
    return ctx.run_dir


def _build_proprag_corpus(ctx, emb, chunk_store, openie_path, tracker, embed_usage, force) -> BenchCorpus:
    graph_path = os.path.join(ctx.cdir, "graph.graphml")
    maps_path = os.path.join(ctx.cdir, "maps.json")
    entity_store = EmbeddingStore(emb, ctx.cdir, "entity")
    proposition_store = EmbeddingStore(emb, ctx.cdir, "proposition")

    with open(openie_path, "r", encoding="utf-8") as f:
        chunk_props = json.load(f)

    if os.path.isfile(graph_path) and os.path.isfile(maps_path) and not force:
        graph = ig.Graph.Read_GraphML(graph_path)
        maps = _load_json(maps_path, {})
        embed_usage.setdefault("PropRAG", {"embed_texts": 0, "wall_time_s": 0.0})
        return BenchCorpus(ctx.corpus_ident, chunk_store, entity_store, proposition_store,
                           graph, maps["proposition_to_entities_map"],
                           maps["chunk_propositions"], ctx.cid2title)

    before = tracker.snapshot("index").as_dict()
    t0 = time.monotonic()
    entities, prop_texts, prop_to_entities = set(), [], {}
    for chunk_id, props in chunk_props.items():
        for prop in props:
            entities.update(prop["entities"])
            pkey = compute_mdhash_id(prop["text"], prefix="proposition-")
            prop_texts.append(prop["text"])
            prop_to_entities[pkey] = prop["entities"]

    entity_store.insert_strings(sorted(entities))
    proposition_store.insert_strings(prop_texts)
    graph = GraphBuilder(ctx.poc_cfg).build(
        chunk_store.get_all_ids(), chunk_props, entity_store, chunk_store
    )
    graph.write_graphml(graph_path)
    _save_json(maps_path, {
        "proposition_to_entities_map": prop_to_entities,
        "chunk_propositions": chunk_props,
    })
    embed_usage["PropRAG"] = {
        "embed_texts": _delta(before, tracker.snapshot("index").as_dict()).get("embed_texts", 0),
        "wall_time_s": round(time.monotonic() - t0, 3),
    }
    return BenchCorpus(ctx.corpus_ident, chunk_store, entity_store, proposition_store,
                       graph, prop_to_entities, chunk_props, ctx.cid2title)


def _load_graphrag_index(ctx, emb, chunk_store, tracker, embed_usage):
    gdir = gr_index._gdir(ctx.cdir)
    entity_store = EmbeddingStore(emb, gdir, gr_index._GR_NAMESPACE)
    edata = _load_json(gr_index._paths(gdir)["entities"], {})
    before = tracker.snapshot("index").as_dict()
    t0 = time.monotonic()
    entity_store.insert_strings(edata.get("embed_texts", []))
    embed_usage["GraphRAG"] = {
        "embed_texts": _delta(before, tracker.snapshot("index").as_dict()).get("embed_texts", 0),
        "wall_time_s": round(time.monotonic() - t0, 3),
    }
    return gr_index._load(gdir, entity_store, chunk_store)


def _retrieve_all(ctx, cfg, retrievers, retr_path, done) -> None:
    for qi, q in enumerate(ctx.questions, 1):
        logger.info("Retrieve Q %d/%d [%s] %s", qi, len(ctx.questions), q.qtype, q.question[:70])
        for system in ctx.systems:
            if (q.qid, system) in done:
                continue
            retriever = retrievers[system]
            try:
                t0 = time.monotonic()
                passages = retriever.retrieve(q.question, top_k=ctx.poc_cfg.retrieval_top_k)
                lat = time.monotonic() - t0
                retrieved_titles = [ctx.cid2title.get(p.chunk_id, "") for p in passages]
                recall = recall_at_k(q.gold_titles, retrieved_titles, cfg.recall_ks)
                if system == "GraphRAG":
                    context = retriever.build_qa_context(q.question, passages)
                else:
                    context = [p.text for p in passages[: cfg.qa_top_k]]
                _append_jsonl(retr_path, {
                    "qid": q.qid, "qtype": q.qtype, "system": system,
                    "question": q.question, "gold_answer": q.answer,
                    "gold_titles": q.gold_titles, "gold_answers": q.gold_answers,
                    "retrieved": [
                        {"chunk_id": p.chunk_id, "title": ctx.cid2title.get(p.chunk_id, ""),
                         "score": p.score} for p in passages
                    ],
                    "recall": recall,
                    "context": context,
                    "retrieval_latency_s": round(lat, 3),
                })
            except Exception as e:  # noqa: BLE001 - one failure must not kill the phase
                logger.exception("Retrieval failed for %s / %s", q.qid, system)
                _append_jsonl(retr_path, {
                    "qid": q.qid, "qtype": q.qtype, "system": system, "error": str(e),
                })


def _merge_index_usage(ctx, embed_usage: Dict[str, Dict]) -> None:
    """Combine Phase A chat usage + Phase B embed usage into report's index_usage."""
    chat = _load_json(os.path.join(ctx.run_dir, "index_chat_usage.json"), {})
    embed = embed_usage or _load_json(os.path.join(ctx.run_dir, "index_embed_usage.json"), {})
    out: Dict[str, Dict] = {}
    for system in ("BaseRAG", "PropRAG", "GraphRAG"):
        c = chat.get(system, {})
        e = embed.get(system, {})
        out[system] = {
            "chat_calls": c.get("chat_calls", 0),
            "chat_prompt_tokens": c.get("chat_prompt_tokens", 0),
            "chat_completion_tokens": c.get("chat_completion_tokens", 0),
            "embed_texts": e.get("embed_texts", 0),
            "wall_time_s": round(c.get("wall_time_s", 0.0) + e.get("wall_time_s", 0.0), 3),
            "parse_failures": c.get("parse_failures", 0),
        }
    _save_json(os.path.join(ctx.run_dir, "index_usage.json"), out)


# ====================================================================== PHASE C
def phase_c_qa(cfg: BenchmarkConfig, make_charts: bool = True) -> str:
    """Chat-LLM-only QA over the stored retrieval contexts, then build the report."""
    ctx = _prepare(cfg)
    retr_path = os.path.join(ctx.run_dir, "retrieval.jsonl")
    rows = [r for r in _read_jsonl(retr_path) if not r.get("error")]
    if not rows:
        raise RuntimeError("Phase C needs Phase B output (retrieval.jsonl). Run --phase b first.")

    store = ResultsStore(ctx.run_dir)
    done = store.done_keys()
    todo = [r for r in rows if (r["qid"], r["system"]) not in done]

    if todo:
        check_backend(ctx.poc_cfg)
        logger.info("Phase C: loading GGUF chat model for QA (%d answers) ...", len(todo))
        llm = BenchLLMClient(ctx.poc_cfg)
        tracker = get_usage_tracker()
        try:
            for i, r in enumerate(todo, 1):
                scope = f"q::{r['system']}"
                before = tracker.snapshot(scope).as_dict()
                t1 = time.monotonic()
                with tracker.scope(scope):
                    answer, raw = answer_question(llm, r["question"], r["context"], cfg)
                qa_latency = time.monotonic() - t1
                usage = _delta(before, tracker.snapshot(scope).as_dict())
                store.append({
                    "qid": r["qid"], "qtype": r["qtype"], "system": r["system"],
                    "question": r["question"], "gold_answer": r.get("gold_answer"),
                    "gold_titles": r["gold_titles"], "answer": answer, "raw_answer": raw,
                    "retrieved": r["retrieved"], "recall": r["recall"],
                    "em": em_score(r["gold_answers"], answer),
                    "f1": f1_score(r["gold_answers"], answer),
                    "retrieval_latency_s": r["retrieval_latency_s"],
                    "qa_latency_s": round(qa_latency, 3),
                    "usage": usage, "ts": time.time(),
                })
                logger.info("QA %d/%d done [%s/%s]", i, len(todo), r["system"], r["qid"])
        finally:
            llm.unload()

    report_mod.build(ctx.run_dir, make_charts=make_charts)
    logger.info("Phase C complete. Report: %s", os.path.join(ctx.run_dir, "report.md"))
    return ctx.run_dir


# ---------------------------------------------------------------------- driver
def _parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Colab VRAM-safe PropRAG/GraphRAG/BaseRAG benchmark")
    p.add_argument("--questions", type=int, default=50)
    p.add_argument("--pilot", type=int, default=None, help="stratified k-of-subset pilot")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--systems", type=str, default="BaseRAG,GraphRAG,PropRAG")
    p.add_argument("--phase", choices=["a", "b", "c", "all"], default="all")
    p.add_argument("--force-reindex", action="store_true")
    p.add_argument("--report-only", action="store_true")
    p.add_argument("--no-charts", action="store_true")
    return p.parse_args(argv)


def _make_cfg(args) -> BenchmarkConfig:
    project_dir = os.environ.get("PROPRAG_PROJECT_DIR", "/content/proprag_run")
    cfg = BenchmarkConfig(
        project_dir=project_dir, n_questions=args.questions, seed=args.seed, pilot=args.pilot,
    )
    cfg.systems = tuple(s.strip() for s in args.systems.split(",") if s.strip())
    return cfg


def main(argv=None) -> None:
    args = _parse_args(argv)
    _setup_logging()
    cfg = _make_cfg(args)

    if args.report_only:
        ctx = _prepare(cfg)
        report_mod.build(ctx.run_dir, make_charts=not args.no_charts)
        print("Report:", os.path.join(ctx.run_dir, "report.md"))
        return

    if args.phase in ("a", "all"):
        phase_a_extract(cfg, force=args.force_reindex)
    if args.phase in ("b", "all"):
        phase_b_embed_retrieve(cfg, force=args.force_reindex)
    if args.phase in ("c", "all"):
        phase_c_qa(cfg, make_charts=not args.no_charts)


if __name__ == "__main__":
    main()


## 6 · Download the dataset (2WikiMultiHopQA)

The two JSON files use the PropRAG/HippoRAG format `dataset.py` expects. Default
source is the PropRAG repo; if the raw download fails (e.g. file too large), set
`USE_HF_DATASET = True` to pull from the `osunlp/HippoRAG_2` dataset, or upload
the two files manually into `DATASET_DIR`.

In [ ]:
import os, urllib.request
DATASET_DIR = "/content/datasets"
os.makedirs(DATASET_DIR, exist_ok=True)
os.environ["PROPRAG_DATASET_DIR"] = DATASET_DIR

USE_HF_DATASET = False           # flip to True to use the HF dataset instead
GH_BASE = "https://raw.githubusercontent.com/ReLink-Inc/PropRAG/main/reproduce/dataset"
FILES = ["2wikimultihopqa.json", "2wikimultihopqa_corpus.json"]

def _have(f): return os.path.isfile(os.path.join(DATASET_DIR, f)) and os.path.getsize(os.path.join(DATASET_DIR, f)) > 0

if not USE_HF_DATASET:
    for f in FILES:
        dst = os.path.join(DATASET_DIR, f)
        if _have(f):
            print("have", f); continue
        print("downloading", f, "...")
        urllib.request.urlretrieve(f"{GH_BASE}/{f}", dst)
        print("  ->", os.path.getsize(dst)//1024, "KB")
else:
    from huggingface_hub import hf_hub_download
    for f in FILES:
        if _have(f): print("have", f); continue
        p = hf_hub_download(repo_id="osunlp/HippoRAG_2", filename=f, repo_type="dataset")
        os.replace(p, os.path.join(DATASET_DIR, f))
        print("fetched", f)

# validate they parse
sys.path.insert(0, "/content")
from benchmark.dataset import load_questions, build_corpus
qs = load_questions(os.path.join(DATASET_DIR, FILES[0]))
print(f"{len(qs)} questions loaded; sample type={qs[0].qtype!r}")

## 7 · Download the gpt-oss-20b GGUF (Q4_K_M)

~12 GB, cached to Drive so it downloads once.

In [ ]:
import os
from huggingface_hub import hf_hub_download
GGUF_REPO = "unsloth/gpt-oss-20b-GGUF"           # <- change repo/file if you prefer
GGUF_FILE = "gpt-oss-20b-Q4_K_M.gguf"
MODEL_DIR = os.path.join(os.environ["PROPRAG_PROJECT_DIR"], "models")
os.makedirs(MODEL_DIR, exist_ok=True)
gguf_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE, local_dir=MODEL_DIR)
os.environ["PROPRAG_GGUF_MODEL_PATH"] = gguf_path
os.environ["PROPRAG_GGUF_LABEL"] = GGUF_FILE
print("GGUF ->", gguf_path, "(", os.path.getsize(gguf_path)//(1024*1024), "MB )")

## 8 · Configure the benchmark environment

Selects the direct-GGUF chat backend, NV-Embed-v2 in 8-bit, and the pilot size.
`llama.cpp` offloads all layers to the GPU (`N_GPU_LAYERS=-1`).

In [ ]:
import os
os.environ["PROPRAG_LLM_BACKEND"]        = "llama_cpp"
os.environ["PROPRAG_EMBEDDING_BACKEND"]  = "sentence_transformers"
os.environ["PROPRAG_EMBEDDING_MODEL"]    = "nvidia/NV-Embed-v2"
os.environ["PROPRAG_EMBED_8BIT"]         = "1"      # 8-bit NV-Embed on the T4
os.environ["PROPRAG_EMBED_MAX_LEN"]      = "512"    # bound seq length (short 2wiki docs)
os.environ["PROPRAG_EMBED_BATCH"]        = "4"      # small batch keeps 8-bit within VRAM
os.environ["PROPRAG_LLAMA_N_GPU_LAYERS"] = "-1"     # all layers on GPU
os.environ["PROPRAG_LLAMA_N_CTX"]        = "8192"

PILOT = 12          # stratified pilot size (10–20 recommended for the free tier)
SEED  = 42
SYSTEMS = "BaseRAG,GraphRAG,PropRAG"

import importlib, benchmark.run_colab as rc, benchmark.bench_config as bcfg
importlib.reload(bcfg); importlib.reload(rc)
rc._setup_logging()
cfg = bcfg.BenchmarkConfig(project_dir=os.environ["PROPRAG_PROJECT_DIR"],
                           n_questions=50, seed=SEED, pilot=PILOT)
cfg.systems = tuple(SYSTEMS.split(","))
print("run dir:", os.path.join(cfg.data_dir, "benchmark"))

## 9 · Phase A — extraction (chat LLM resident)

Loads the GGUF model and runs NER + propositions (PropRAG) and entity/relation
extraction + community reports (GraphRAG). Unloads the LLM at the end. On a T4
this is the slow part; it is fully cached, so a re-run resumes instantly.

In [ ]:
rc.phase_a_extract(cfg)

## 10 · Phase B — embeddings + retrieval (embedder resident)

Loads NV-Embed-v2 in 8-bit, builds every vector store + graph, then runs the
full retrieval for all questions × systems (no LLM) and scores Recall@k.
Unloads the embedder at the end.

> If VRAM wasn't fully freed after Phase A, use **Runtime → Restart runtime**,
> re-run cells 4–8, then run this cell — all Phase A work is already on Drive.

In [ ]:
rc.phase_b_embed_retrieve(cfg)

## 11 · Phase C — QA + report (chat LLM resident)

Reloads the GGUF model and answers every question from its stored context,
scores EM/F1, writes `results.jsonl` and builds the report.

In [ ]:
run_dir = rc.phase_c_qa(cfg, make_charts=True)
print("run dir:", run_dir)

## 12 · Show the report

In [ ]:
import os
from IPython.display import Markdown, Image, display
with open(os.path.join(run_dir, "report.md"), encoding="utf-8") as f:
    display(Markdown(f.read()))
for name in ("quality.png", "index_tokens.png"):
    p = os.path.join(run_dir, "charts", name)
    if os.path.isfile(p):
        display(Image(filename=p))